<a href="https://colab.research.google.com/github/ABROCK1516/Ukraine-Losses-Weather/blob/main/Ukraine_Losses_Weather.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Russian Equipment & Personnel Losses & Front-line Weather**
## *Data Acquisition and Preprocessing – Individual project*
**Alex Brock | C23426432 | TU912/4**

## **Introduction**
This project combines daily figures for Russian equipment and personnel losses in the war in Ukraine with weather observations from near the front line. Weather is often said to shape the fighting, for example mud seasons slowing armoured vehicles, or wind and fog grounding FPV drones which allows for more ground assaults and offensive actions. This project tests whether those patterns can be seen in the data.

I chose this topic because I have been following the war closely since the full scale invasion of Ukraine in 2022. I keep up to date with the war through YouTube channels and the news. From the start of 2023 onwards, I have watched and read about both sides of the war using weather patterns and conditions to their advantage such as fog to aid in assaulting dug-in positions.

I was particularly interested in the military side: how weapons and drones are used, and how much outside conditions like the weather affect them. Commentary on the war often claims that weather changes the fighting, and I wanted to see whether the data actually supports that.

## **Research Questions**
The project is built in three layers, each adding a dataset and new questions:

1. **Kaggle data alone:** How have daily losses changed over the course of the war? Has the mix of equipment lost changed, and has the ratio of personnel to vehicles lost changed over time?
2. **Adding weather:** Do daily losses vary with weather conditions such as rain, freezing temperatures, wind and low visibility? Which loss categories are most affected?
3. **Adding combat engagements (extension):** When losses change with the weather, is that because there is less fighting, or because the fighting causes fewer losses?

## **Data Sources**

| Source | What it contains | Period | How it was obtained | Role |
|---|---|---|---|---|
| [Kaggle: 2022 Russia Ukraine War (piterfm)](https://www.kaggle.com/datasets/piterfm/2022-ukraine-russian-war) | Daily cumulative Russian equipment losses by category, and personnel losses | 25 Feb 2022 – 10 Oct 2026 (1,689 rows) | Downloaded as two CSV files | Main dataset |
| [pogodaiklimat.ru](http://www.pogodaiklimat.ru/) | 3-hourly weather observations: wind, visibility, cloud cover, weather phenomena, temperature, precipitation | Jan 2023 – Oct 2026 | Web scraping (urllib and BeautifulSoup) | Complementary dataset |
| [ukraine.vefgreining.com](https://ukraine.vefgreining.com/) | Daily number of combat engagements reported by Ukraine's General Staff | 22 Feb 2023 – 16 Oct 2026 | JSON data file | Extension |

Weather is taken from two stations: **occupied Donetsk** (station 34519), which is close to most of the heaviest fighting since 2023, and **Kharkiv** (station 34300), in the Kharkiv region, where fighting continues around Kupiansk, it is used as a second station to check whether results hold.

## **Notes on the data**
- **These are claimed figures.** The loss and engagement figures are reported by Ukraine and have not been independently verified.
- **Personnel figures are estimates.** The dataset itself marks them as approximate (“about” or “more”).
- **"Occupied Donetsk".** The Donetsk weather station is in Russian-occupied territory, and the weather site lists it under Russia.
- **Weather coverage.** The weather site has little data for 2022 (spot-checked in Step 2: both stations have no readings in February or July 2022, and Donetsk only resumes on 18 November 2022, with about 18% of readings missing), so the weather analysis covers 2023 onwards. The Kaggle data is kept in full for Step 1
- **One weather point for a long front.** The front line is over 1,000 km long, so weather at one station is only an approximation of conditions where the fighting took place.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# Import the os library, which lets Python work with folders and file paths
import os
os.chdir("/content/drive/MyDrive/Data ACQ - Project/Data")
!pwd

/content/drive/MyDrive/Data ACQ - Project/Data


## **Step 1: Import the Kaggle dataset**
The dataset is "2022 Russia Ukraine War" by piterfm on Kaggle. It has two CSV files:
running totals of Russian equipment losses and personnel losses.

**About the dataset**

**Goal:** load the Kaggle data, understand its structure, columns, types and values, and turn the running totals into daily losses.

**Assignment:** Step 1, importing and assessing the Kaggle dataset **(20 marks).**

**Research link:** daily losses are the main measure in all three research questions.

The main dataset is ["2022 Russia Ukraine War" by piterfm](https://www.kaggle.com/datasets/piterfm/2022-ukraine-russian-war) on Kaggle. It records Russian losses in the war as reported by official Ukrainian sources, in two CSV files:
- `russia_losses_equipment.csv` (1,689 rows × 20 columns): losses by equipment category, such as tanks, armoured personnel carriers (APCs), artillery and drones.
- `russia_losses_personnel.csv` (1,689 rows × 5 columns): personnel losses and prisoners of war (POW).

Both files cover 25 February 2022 to 10 October 2026, with one row per day.

The figures are **cumulative**: each row gives the total lost since the start of the invasion, not the losses on that day, so daily losses have to be calculated. The data was downloaded on 16 October 2026. The dataset is updated daily, so it is not downloaded again, which keeps the results reproducible.

### **1.1 - Loading the Two Files**
I load both CSV files into pandas DataFrames, one called `equipment` and one called `personnel`.

In [3]:
# Import pandas, the library used to load and work with tables of data
import pandas as pd

# Load the equipment losses CSV into a DataFrame (a table) called equipment
equipment = pd.read_csv("russia_losses_equipment.csv")

# Load the personnel losses CSV into a DataFrame called personnel
personnel = pd.read_csv("russia_losses_personnel.csv")

### **1.2 - First look: The Equipment File**
**Four checks:** The size and first rows, the data type of each column, summary statistics, and the number of missing values in each column.

In [4]:
print(equipment.shape)  # size of the table: (rows, columns)
equipment.head(10)  # first 10 rows, to see what the data looks like

(1689, 20)


,date,day,aircraft,helicopter,tank,APC,field artillery,MRL,military auto,fuel tank,drone,naval ship,anti-aircraft warfare,special equipment,mobile SRBM system,greatest losses direction,vehicles and fuel tanks,cruise missiles,submarines,ground robotic systems
0,2026-10-10,1690,443,356,12406,25515,51000,2201,NaN,NaN,556926,35,1699,4812.0,NaN,NaN,160946.0,5126.0,2.0,2917.0
1,2026-10-09,1689,443,356,12402,25509,50950,2199,NaN,NaN,555335,35,1693,4810.0,NaN,NaN,160412.0,5126.0,2.0,2908.0
2,2026-10-08,1688,443,356,12399,25502,50883,2194,NaN,NaN,553578,35,1692,4806.0,NaN,NaN,159795.0,5126.0,2.0,2890.0
3,2026-10-07,1687,443,356,12396,25498,50836,2192,NaN,NaN,551876,35,1690,4800.0,NaN,NaN,159128.0,5126.0,2.0,2878.0
4,2026-10-06,1686,443,356,12396,25494,50785,2184,NaN,NaN,549986,35,1688,4795.0,NaN,NaN,158551.0,5126.0,2.0,2866.0
5,2026-10-05,1685,443,356,12393,25487,50733,2182,NaN,NaN,548095,35,1688,4793.0,NaN,NaN,157985.0,5126.0,2.0,2859.0
6,2026-10-04,1684,443,356,12391,25483,50686,2179,NaN,NaN,546240,35,1686,4788.0,NaN,NaN,157258.0,5126.0,2.0,2837.0
7,2026-10-03,1683,443,356,12391,25477,50637,2178,NaN,NaN,544498,35,1683,4783.0,NaN,NaN,156731.0,5126.0,2.0,2824.0
8,2026-10-02,1682,443,356,12388,25471,50592,2175,NaN,NaN,542853,35,1683,4780.0,NaN,NaN,156309.0,5126.0,2.0,2810.0
9,2026-10-01,1681,443,356,12388,25464,50555,2174,NaN,NaN,541383,35,1683,4774.0,NaN,NaN,155887.0,5126.0,2.0,2803.0


In [5]:
# Show the column names and the data type of each column
equipment.dtypes

,0
date,object
day,int64
aircraft,int64
helicopter,int64
tank,int64
APC,int64
field artillery,int64
MRL,int64
military auto,float64
fuel tank,float64


In [6]:
# Summary statistics for each numeric column:
equipment.describe()

,day,aircraft,helicopter,tank,APC,field artillery,MRL,military auto,fuel tank,drone,naval ship,anti-aircraft warfare,special equipment,mobile SRBM system,vehicles and fuel tanks,cruise missiles,submarines,ground robotic systems
count,1689.000000,1689.000000,1689.000000,1689.000000,1689.000000,1689.000000,1689.000000,65.000000,65.000000,1689.000000,1689.000000,1689.000000,1670.000000,36.000000,1624.000000,1624.000000,1123.000000,161.000000
mean,846.000000,343.914742,303.917703,7302.714624,14825.561279,17909.660746,1046.749556,1047.507692,69.323077,66177.216104,23.708111,788.043813,2282.670060,3.944444,37143.435961,2428.223522,1.266251,2036.956522
std,487.716619,79.985283,60.524649,3836.696267,7956.872953,15393.569483,548.811027,466.162060,7.545917,125776.097509,7.141723,473.888851,1695.212102,0.333333,38493.183090,1501.023468,0.442194,475.056514
min,2.000000,10.000000,7.000000,80.000000,516.000000,49.000000,4.000000,100.000000,60.000000,0.000000,2.000000,0.000000,10.000000,2.000000,1796.000000,84.000000,1.000000,1306.000000
25%,424.000000,308.000000,294.000000,3675.000000,7131.000000,2837.000000,539.000000,600.000000,60.000000,2402.000000,18.000000,289.000000,381.000000,4.000000,6452.750000,1181.250000,1.000000,1636.000000
50%,846.000000,359.000000,326.000000,7974.000000,15307.000000,13959.000000,1104.000000,1178.000000,73.000000,11187.000000,28.000000,853.000000,2425.500000,4.000000,21002.000000,2401.000000,1.000000,1981.000000
75%,1268.000000,421.000000,340.000000,11104.000000,23130.000000,31458.000000,1466.000000,1437.000000,76.000000,51043.000000,28.000000,1207.000000,3942.750000,4.000000,60242.750000,3635.500000,2.000000,2442.000000
max,1690.000000,443.000000,356.000000,12406.000000,25515.000000,51000.000000,2201.000000,1701.000000,76.000000,556926.000000,35.000000,1699.000000,4812.000000,4.000000,160946.000000,5126.000000,2.000000,2917.000000


In [7]:
# Count the missing (empty) values in each column
equipment.isnull().sum()

,0
date,0
day,0
aircraft,0
helicopter,0
tank,0
APC,0
field artillery,0
MRL,0
military auto,1624
fuel tank,1624


### **1.3 - First Look: The Personnel File**
**Four checks:** The size and first rows, the data type of each column, summary statistics, and the number of missing values in each column.

In [8]:
print(personnel.shape)  # size of the table: (rows, columns)
personnel.head()  # first 5 rows

(1689, 5)


,date,day,personnel,personnel*,POW
0,2026-10-10,1690,1550910,about,NaN
1,2026-10-09,1689,1549490,about,NaN
2,2026-10-08,1688,1547780,about,NaN
3,2026-10-07,1687,1545950,about,NaN
4,2026-10-06,1686,1544260,about,NaN


In [9]:
# Show the column names and the data type of each column
personnel.dtypes

,0
date,object
day,int64
personnel,int64
personnel*,object
POW,float64


In [10]:
# Summary statistics for each numeric column:
personnel.describe()

,day,personnel,POW
count,1689.000000,1.689000e+03,62.000000
mean,846.000000,6.250922e+05,386.387097
std,487.716619,4.793099e+05,131.440363
min,2.000000,2.800000e+03,0.000000
25%,424.000000,1.864200e+05,389.000000
50%,846.000000,5.286200e+05,421.000000
75%,1268.000000,1.067100e+06,474.500000
max,1690.000000,1.550910e+06,496.000000


In [11]:
# Count the missing (empty) values in each column
personnel.isnull().sum()

,0
date,0
day,0
personnel,0
personnel*,0
POW,1627


### **1.4 - Do the Two Files Line up?**
The two files will be joined on `date`, so I check they have the same date in every row. `.all()` returns `True` only if every row matches.

In [12]:
# Compare the date columns row by row; .all() is True only if every row matches
(equipment["date"] == personnel["date"]).all()

np.True_

### **1.5 - When was Each Category Reported?**
Several columns have many missing values. For each one, I print the first and last date with data and the number of filled rows, to see whether the gaps are errors or changes in what was reported. I do the same for `POW` in the personnel file.

In [13]:
# Columns with missing values, to check when each one has data
columns_to_check = ["military auto", "fuel tank", "vehicles and fuel tanks",
                    "cruise missiles", "submarines", "ground robotic systems",
                    "mobile SRBM system", "greatest losses direction",
                    "special equipment"]  # special equipment has 19 missing values

for col in columns_to_check:  # go through each column name in the list
    filled = equipment[equipment[col].notnull()]  # keep only the rows where this column is NOT empty
    print(col, "|", filled["date"].min(), "to", filled["date"].max(), "|", len(filled), "rows")  # first date, last date, rows with data

military auto | 2022-02-25 to 2022-04-30 | 65 rows
fuel tank | 2022-02-25 to 2022-04-30 | 65 rows
vehicles and fuel tanks | 2022-05-01 to 2026-10-10 | 1624 rows
cruise missiles | 2022-05-01 to 2026-10-10 | 1624 rows
submarines | 2023-09-14 to 2026-10-10 | 1123 rows
ground robotic systems | 2026-05-03 to 2026-10-10 | 161 rows
mobile SRBM system | 2022-03-26 to 2022-04-30 | 36 rows
greatest losses direction | 2022-04-25 to 2023-01-02 | 203 rows
special equipment | 2022-03-16 to 2026-10-10 | 1670 rows


In [14]:
# Keep only the rows where POW has a value
pow_filled = personnel[personnel["POW"].notnull()]
# Print the first date, last date and number of rows with a POW value
print(pow_filled["date"].min(), "to", pow_filled["date"].max(), "|", len(pow_filled), "rows")

2022-02-25 to 2022-04-27 | 62 rows


### **1.6 - Observaions on the Kaggle Dataset**
**Structure:** Each row is one day, and the rows are in newest-first order (the first row is 10 October 2026). The figures are running totals since the start of the invasion, not the losses on that day. The equipment file has 1,689 rows and 20 columns, and the personnel file has 1,689 rows and 5 columns. Both files have the same date in every row (the check returned `True`), so they can be joined on `date`.

**Data types:** `date` is stored as `object`, which means pandas treats it as text, so it needs converting before it can be sorted or compared as a date. Some whole-number columns are stored as decimals (`float64`) because they contain empty values, which a whole-number column can't hold.

**Values:**
- `day` starts at 2. The invasion began on **24 February 2022 (day 1)**, so the first row (25 February) already covers two days of losses.
- Because the figures are running totals, each **column's minimum is just its first value**. For example, 516 APCs had already been claimed in the first row.
- `drone` and `anti-aircraft warfare` start at 0. Drones were used from the start of the invasion, so these categories probably **weren't being counted** yet.
- Some categories start high when they first appear: `cruise missiles` at 84 and `ground robotic systems` at 1,306. This suggests earlier losses were added in one go on the first day. Since ground robotic systems were first reported (3 May 2026).

**Missing values:** The gaps aren't errors. They come from changes in which categories the daily report included:
- **Replaced:** `military auto` and `fuel tank` stop on 30 April 2022, and `vehicles and fuel tanks` starts the next day. Together they fill every row (65 + 1,624 = 1,689), so the **two old categories were combined into one**.
- **Added later:** `special equipment` (16 March 2022: its 19 missing values are the first 19 days of the war), `cruise missiles` (1 May 2022), `submarines` (14 September 2023) and `ground robotic systems` (3 May 2026). Submarines first appear the day after Ukraine's missile strike on the submarine *Rostov-on-Don* in Sevastopol on 13 September 2023 ([Naval News](https://www.navalnews.com/?p=58386)). This fits each day's report covering the day before.
- **Dropped:** `mobile SRBM system` was only reported from 26 March to 30 April 2022. `greatest losses direction` contains place names, not numbers, and was only reported from 25 April 2022 to 2 January 2023, with gaps, so it is not used.
- **Personnel:** `POW` was only reported from 25 February to 27 April 2022. After that it stopped being reported; the data doesn't say why - **possibly the data was confidential.**

So each category can only be compared over the period it was reported.



### **1.7 From Running Totals to Daily Losses**
The figures are running totals, but the research questions are about losses on each day. To get daily losses, I:

**1.** Convert `date` from text to real dates, so pandas can sort and compare them

**2.** Join the equipment and personnel files into one table on `date`

**3.** Sort the rows from oldest to newest (the files are newest first)

**4.** Subtract the previous day's total from each day's total

#### **1.7.1 - Converting the dates**

In [15]:
equipment["date"] = pd.to_datetime(equipment["date"])  # convert equipment dates from text to real dates
personnel["date"] = pd.to_datetime(personnel["date"])  # same for personnel

print(equipment["date"].dtype)  # should now show datetime64 instead of object
print(personnel["date"].dtype)  # same check for personnel

datetime64[ns]
datetime64[ns]


#### **1.7.2 - Joining the Two Files**

Both files have the same 1,689 dates **(checked in 1.4)**, so I join them on `date` with `pd.merge`. I use `how="outer"`, which keeps dates found in either file, so any mismatch would show up as empty values instead of being **silently dropped**. I leave out the personnel file's `day` column, because the equipment file already has it.

In [16]:
personnel_cols = personnel[["date", "personnel", "personnel*", "POW"]]  # keep only the columns needed (double brackets = a list of columns)

losses = pd.merge(equipment, personnel_cols, on="date", how="outer")  # join on date; "outer" keeps dates found in either file, so mismatches would show as empty values

losses.shape  # expect 1,689 rows and 23 columns (20 from equipment + 3 from personnel)

(1689, 23)

#### **1.7.3 - Sorting by Date**
`.sort_values("date")` puts the rows in order from oldest to newest, so each row follows the day before it. `.reset_index(drop=True)` renumbers the rows 0, 1, 2… in the new order.

In [17]:
losses = losses.sort_values("date")  # oldest first, so each row follows the day before it
losses = losses.reset_index(drop=True)  # renumber the rows; drop=True throws away the old row numbers

losses.head()  # the first row should now be 25 Feb 2022

,date,day,aircraft,helicopter,tank,APC,field artillery,MRL,military auto,fuel tank,...,special equipment,mobile SRBM system,greatest losses direction,vehicles and fuel tanks,cruise missiles,submarines,ground robotic systems,personnel,personnel*,POW
0,2022-02-25,2,10,7,80,516,49,4,100.0,60.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2800,about,0.0
1,2022-02-26,3,27,26,146,706,49,4,130.0,60.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4300,about,0.0
2,2022-02-27,4,27,26,150,706,50,4,130.0,60.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4500,about,0.0
3,2022-02-28,5,29,29,150,816,74,21,291.0,60.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5300,about,0.0
4,2022-03-01,6,29,29,198,846,77,24,305.0,60.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5710,about,200.0


#### **Checking for Missing Days**
`.diff()` on the dates gives the gap between each date and the one before it, and `.value_counts()` counts how often each gap size appears. If no days are missing or repeated, every gap is 1 day.

In [18]:
losses["date"].diff().value_counts()  # gap between each date and the one before; every gap should be 1 day

,count
date,
1 days,1688


#### **1.7.4 - Calculating daily losses**

In [19]:
# The running-total columns to turn into daily losses
loss_columns = ["aircraft", "helicopter", "tank", "APC", "field artillery", "MRL",
                "drone", "naval ship", "anti-aircraft warfare", "special equipment",
                "vehicles and fuel tanks", "cruise missiles", "submarines",
                "ground robotic systems", "personnel"]

for col in loss_columns:  # go through each column in the list
    losses["daily " + col] = losses[col].diff()  # today's total minus yesterday's = losses claimed that day

losses[["date", "tank", "daily tank", "personnel", "daily personnel"]].head(10)  # check a few columns side by side

,date,tank,daily tank,personnel,daily personnel
0,2022-02-25,80,NaN,2800,NaN
1,2022-02-26,146,66.0,4300,1500.0
2,2022-02-27,150,4.0,4500,200.0
3,2022-02-28,150,0.0,5300,800.0
4,2022-03-01,198,48.0,5710,410.0
5,2022-03-02,211,13.0,5840,130.0
6,2022-03-03,217,6.0,9000,3160.0
7,2022-03-04,251,34.0,9166,166.0
8,2022-03-05,269,18.0,10000,834.0
9,2022-03-06,285,16.0,11000,1000.0


### **1.8 - Checking the Daily Figures**
A running total should never go down, so a negative daily figure would mean the source corrected an earlier number.

In [20]:
for col in loss_columns:  # go through each loss category
    negatives = (losses["daily " + col] < 0).sum()  # count the days where the daily figure is below zero
    print(col, negatives)  # print the category and its count

aircraft 2
helicopter 0
tank 1
APC 1
field artillery 1
MRL 2
drone 0
naval ship 0
anti-aircraft warfare 2
special equipment 2
vehicles and fuel tanks 0
cruise missiles 2
submarines 0
ground robotic systems 0
personnel 0


#### **Summary Statistics for the Main Categories**
`.describe()` shows the typical daily loss (the median, in the 50% row) and the range for the categories used most in the analysis.

In [21]:
# Summary statistics for the daily figures of the main categories
losses[["daily tank", "daily APC", "daily field artillery", "daily drone", "daily personnel"]].describe()

,daily tank,daily APC,daily field artillery,daily drone,daily personnel
count,1688.000000,1688.000000,1688.000000,1688.000000,1688.000000
mean,7.302133,14.809834,30.184242,329.932464,917.126777
std,6.535839,13.564484,22.665864,626.922771,420.982929
min,-2.000000,-14.000000,-10.000000,0.000000,0.000000
25%,3.000000,5.000000,10.000000,8.000000,600.000000
50%,5.000000,11.000000,26.000000,36.000000,970.000000
75%,10.000000,20.000000,46.000000,205.250000,1210.000000
max,66.000000,190.000000,122.000000,2719.000000,3160.000000


#### **The Negative and Zero Days**
For the three main categories with a negative minimum, I list the dates of the negative days. I also list the days where no personnel losses were added.

In [22]:
for col in ["tank", "APC", "field artillery"]:  # the columns with negative minimums
    neg_days = losses[losses["daily " + col] < 0]  # keep only the rows where the daily figure is negative
    print(neg_days[["date", "daily " + col]])  # show the date and the negative value

zero_personnel = losses[losses["daily personnel"] == 0]  # rows where no personnel losses were added
print(zero_personnel[["date", "personnel", "daily personnel"]])  # show when they happened

          date  daily tank
457 2023-05-28        -2.0
          date  daily APC
230 2022-10-13      -14.0
           date  daily field artillery
1468 2026-03-04                  -10.0
         date  personnel  daily personnel
10 2022-03-07      11000              0.0
12 2022-03-09      12000              0.0
13 2022-03-10      12000              0.0
14 2022-03-11      12000              0.0
15 2022-03-12      12000              0.0
16 2022-03-13      12000              0.0
17 2022-03-14      12000              0.0


#### **The Tank Correction in Context**
I show three days either side of the −2 tank day, to see whether it cancels out an unusually high day before it.

In [23]:
losses[["date", "tank", "daily tank"]][454:461]  # rows 454 to 460: three days either side of the tank correction

,date,tank,daily tank
454,2023-05-25,3795,3.0
455,2023-05-26,3796,1.0
456,2023-05-27,3799,3.0
457,2023-05-28,3797,-2.0
458,2023-05-29,3801,4.0
459,2023-05-30,3802,1.0
460,2023-05-31,3802,0.0


### **1.9 - Observations on Daily Losses**
**Dates:** After sorting, every gap between dates is exactly one day (1,688
gaps for 1,689 rows), so no days are missing or repeated.

**Negative days:** A running total should never go down, but it does on 13 days across 8 columns. **Aircraft, MRL, anti-aircraft warfare, special equipment and cruise missiles go down twice each, and tank, APC and field artillery once each** (APC −14 on 13 October 2022, tank −2 on 28 May 2023, field artillery −10 on 4 March 2026). These look like **corrections**, where the source lowered a total it had reported earlier. For the tank, the days either side are normal (+3 and +4), so it doesn't look like it is cancelling out one unusually high day. Drones, vehicles and personnel never go down. The negative days are rare and small, but they need handling before the analysis, because a negative daily loss isn't possible.

**Early figures are uneven:** The largest daily tank figure (66, on 26 February 2022) and the largest daily personnel figure (3,160, on 3 March 2022) both come from the first ten days of the war. Seven days in early March 2022 (7 March and 9–14 March) show zero personnel losses, while the totals sit at round numbers (11,000 and 12,000). This suggests the e**arly estimates were rounded and updated every few days**, so early daily figures show when the estimate was updated rather than when the losses happened. This doesn't affect the weather analysis, which starts in January 2023.

**Drones vs tanks:** The median daily drone loss (36) is much higher than for tanks (5), which fits drones being used in far greater numbers and being cheaper to lose. The drone mean (330) is about nine times the median, because a small number of very large days pull it up. Drone use **grew sharply over the war**, so a whole-war average mixes very different periods. When plotting, drones will need a log scale or a rolling average, and the years compared separately.

## **Step 2: Scraping the Weather Data**
**Goal:** scrape 3-hourly weather observations for two stations near the front line, from January 2023 to October 2026, and turn them into one row per day.

**Assignment:** Step 2, scraping a complementary dataset **(40 marks)**.

**Research link:** adds the weather columns needed for **research question 2**: do daily losses vary with the weather?

The data comes from [pogodaiklimat.ru](http://www.pogodaiklimat.ru), a Russian weather archive that publishes weather station readings every 3 hours (8 per day). I scrape two stations: **occupied Donetsk** (34519) and **Kharkiv** (34300).

**This step has eleven parts:**

**1.** Why this site, and how it links to the Kaggle data

**2.** Scraping ethics

**3.** Fetching and inspecting one page

**4.** The bot parameter

**5.** Parsing one month

**6.** Testing before the full scrape

**7.** Scraping every month

**8.** Checking coverage

**9.** Parsing every page

**10.** Cleaning

**11.** One row per day

### **2.1 - Why This Site and How It Links to the Kaggle Data**
**Why this site:** Commentary on the war links the weather to different kinds of fighting: rain and mud slowing tanks and armoured vehicles, frozen ground letting them move again, and wind and fog grounding drones allowing for more troop movement. Testing these needs more than daily averages. This site records, at every 3-hourly reading, precipitation, temperature (with daily minimum and maximum), snow depth, weather phenomena (rain, snow, fog), visibility, wind speed and gusts, and cloud cover.

**New columns:** The Kaggle dataset only records losses. The weather data adds wind speed and gusts, visibility, weather phenomena, cloud cover, temperature and precipitation, so I can ask whether daily losses change with the weather, and which loss categories are most affected.

**How many Kaggle rows it covers:** The weather analysis starts on 1 January 2023 (see the 2022 check in 2.6). From then to 10 October 2026 is 1,379 of the 1,689 Kaggle days (82%), far above the 10% required. Donetsk has no readings on 63 of those days (see 2.8), which leaves about 1,316 days (78%). The exact number of matched rows is counted after the merge in Step 3.

**The common column:** Both datasets have a date, so they can be joined on it directly, with no bridging dataset needed. The weather dates need preparing first:
- the site gives the date as day.month with no year, so the year is added from the file name
- times are in UTC, so they are converted to local time before deciding which day a reading belongs to
- there are 8 readings per day, so they are combined into one row per day, to match Kaggle's one row per day
- Kaggle dates each figure by the day it was reported (the morning after the fighting), so Kaggle dates are shifted back one day before the merge

### **2.2 - Scraping Ethics**
Before requesting any weather data, I check what the site allows: its robots.txt file and its terms of use. I then summarise how the scraper follows them.

#### **Checking robots.txt**
robots.txt tells automated programs which parts of a site they shouldn't access (week 2 ethics). I fetch it with the same `urllib` method used for the weather pages.

In [24]:
import urllib.request  # module from week 2 for fetching web pages

robots_url = "http://www.pogodaiklimat.ru/robots.txt"  # the site's rules for automated programs

request = urllib.request.Request(robots_url)  # build the request
request.add_header("User-Agent", "Mozilla/5.0 (student project, TU Dublin)")  # identify the request politely
opener = urllib.request.build_opener()  # object that sends requests
response = opener.open(request)  # send the request
robots_text = response.read().decode("utf-8")  # read the bytes and turn them into text

print(robots_text)  # display the rules

User-agent: *
Disallow: /admin/
Disallow: /metar/
Disallow: /photo/
Disallow: /synop/
Disallow: /summary/
Disallow: /usasummary/
Disallow: /extremal.php?
Disallow: /extremal1.php?
Disallow: /dextremal.php?
Disallow: /dextremal1.php?

User-agent: Yandex
Disallow: /admin/
Disallow: /metar/
Disallow: /photo/
Disallow: /synop/
Disallow: /summary/
Disallow: /usasummary/
Disallow: /extremal.php?
Disallow: /extremal1.php?
Disallow: /dextremal.php?
Disallow: /dextremal1.php?
Host: www.pogodaiklimat.ru


#### **Checking the Terms of Use**
I fetch the site's user agreement and print only the clauses that apply to scraping and publishing data (3.1 and 3.5–3.7), rather than the whole page. My translation is in the next cell.

In [25]:
from bs4 import BeautifulSoup # library from week 3 for searching through HTML

agreement_url = "http://www.pogodaiklimat.ru/agreement.php"  # the site's user agreement page

request = urllib.request.Request(agreement_url)  # build the request
request.add_header("User-Agent", "Mozilla/5.0 (student project, TU Dublin)")  # identify the request
response = opener.open(request)  # send it, reusing the opener from the robots.txt cell
agreement_html = response.read().decode("utf-8")  # read the bytes and turn them into text

agreement_soup = BeautifulSoup(agreement_html, "html.parser")  # turn the HTML into a searchable object
agreement_text = agreement_soup.get_text()  # keep only the text, without the HTML tags

start_3_1 = agreement_text.find("3.1 ")  # position where clause 3.1 starts (the space stops it matching 3.10)
end_3_1 = agreement_text.find("3.2 ")  # position where clause 3.2 starts, i.e. where 3.1 ends
print(agreement_text[start_3_1:end_3_1])  # print clause 3.1 only

start_3_5 = agreement_text.find("3.5 ")  # position where clause 3.5 starts
end_3_7 = agreement_text.find("3.8 ")  # position where clause 3.8 starts, i.e. where 3.7 ends
print(agreement_text[start_3_5:end_3_7])  # print clauses 3.5, 3.6 and 3.7

3.1 Сайт предназначен для использования исключительно в личных некоммерческих целях. Пользователь обязан пользоваться Сайтом в порядке, определенном Соглашением.

3.5 Полное или частичное копирование, воспроизведение, распространение, размещение в свободном доступе в сети Интернет текстовых и графических материалов Сайта, в том числе - элементов дизайна, любое их использование в средствах массовой информации и/или в коммерческих целях без предварительного письменного разрешения Администрации Сайта запрещено.
3.6 Запрещено копирование, распространение, размещение в свободном доступе в сети Интернет Данных, находящихся в разделе Саммари, недоступном для Пользователей без предварительной авторизации.
3.7 При размещении в свободном доступе Данных с Сайта в сети Интернет обязательно необходима активная ссылка на источник информации (Источник: сайт «Погода и Климат»).



### **Translation of the Printed Clauses:**

**3.1** The site is intended for use solely for personal, non-commercial purposes. Users must use the site in accordance with the rules set out in the Agreement.

**3.5** Full or partial copying, reproduction, distribution, posting in public access on the Internet of the site's text and graphic materials, including design elements, as well as any use in the media and/or for commercial purposes without prior written permission from the Site Administration is prohibited.

**3.6** Copying, distributing, or posting in public access on the Internet data from the Summary section, which is not available to users without prior authorization, is prohibited.

**3.7** When posting data from the site in public access on the Internet, an active link to the information source (Source: "Weather and Climate" website) is required.

*(checked with google translate)*

#### **Ethics Summary**
**Robots.txt:** The rules for all user agents (`User-agent: *`) apply to my scraper. `/weather.php`, the only page I request for data, is not disallowed, and the file sets no crawl delay. I don't access any disallowed path, such as `/summary/`.

**Terms of use:**
- The site is for personal, non-commercial use only **(clause 3.1)**. This project is non-commercial coursework.
- The site's own text, graphics and design may not be copied or republished without permission **(clause 3.5)**, so the raw HTML pages stay private in my Google Drive, and only the extracted weather data goes on GitHub.
- Data from the members-only Summary section may not be shared **(clause 3.6)**. I don't use it.
- When the site's data is published online, a link to the source is required **(clause 3.7)**. The data is credited below and in the README.

**Limiting requests:** Each page is downloaded once and saved to Drive, and every later step reads the saved files, not the live site. Every download cell checks whether the page is already saved before sending a request, and there is a 3-second pause between requests, so the server never gets a burst of load.

**Necessary data only:** Only the two stations and the months needed for the analysis are requested. The pages contain public weather observations only; no login is used and no personal data is collected.

**Identifying the scraper:** Each request sends a User-Agent header that identifies it as a student project, rather than hiding what it is.

**The `bot` parameter:** The site's form asks whether the visitor is a robot, an alien or a human. Answering "robot" returns only the most recent week, whatever dates are requested (see 2.4), so the scraper answers "human" (`bot=2`, the form's default). Because this question acts as a check on automated visitors, I asked my lecturer whether this was acceptable, and they approved it on 23 October 2026.

**Source:** Weather data from [Pogoda i Klimat](http://www.pogodaiklimat.ru) (Источник: сайт «Погода и Климат»).

### **2.3 - Fetching and Inspecting One Page**
Before scraping every month, I look at one page in detail (occupied Donetsk, March 2024) to see how it is built. The site has one page per station per month, and the URL sets what is shown:

`http://www.pogodaiklimat.ru/weather.php?id=34519&bday=1&fday=31&amonth=3&ayear=2024&bot=2`

- `id`: the station number
- `bday` and `fday`: the first and last day to show (`fday` must be a real day, see 2.6)
- `amonth` and `ayear`: the month and year
- `bot`: the answer to the site's "robot / alien / human" question (see 2.4)

This section has three steps:

**1.** Fetch and save one page

**2.** Read and decode it

**3.** Find the tables that hold the readings

#### **2.3.1 - Fetching and Saving One Page**
I save the raw HTML to Drive straight away, so the page never needs downloading again. The cell first checks whether the file is already saved, so re-running it sends no request. `os.path.getsize()` gives the saved file's size in bytes.

In [26]:
import os  # for working with folders (already imported in setup; repeated so this cell works on its own)

# URL for one page: station 34519 (occupied Donetsk), days 1-31, March 2024
url = "http://www.pogodaiklimat.ru/weather.php?id=34519&bday=1&fday=31&amonth=3&ayear=2024&bot=2"
file_name = "raw_html/donetsk_2024_03.html"  # where the page is saved in Drive

os.makedirs("raw_html", exist_ok=True)  # create the folder if it isn't there yet; no error if it exists

if os.path.exists(file_name):  # already saved on an earlier run
    print("Already saved, not downloaded again")  # no request is sent
else:  # first run only
    request = urllib.request.Request(url)  # build the request
    request.add_header("User-Agent", "Mozilla/5.0 (student project, TU Dublin)")  # identify the request
    raw_html = opener.open(request).read()  # send it and read the page as raw bytes
    f = open(file_name, "wb")  # open a new file for writing bytes
    f.write(raw_html)  # save the raw page to Drive
    f.close()  # close the file so it's fully saved
    print("Downloaded and saved")

print(os.path.getsize(file_name), "bytes")  # size of the saved file; first run gave 228,826

Already saved, not downloaded again
228826 bytes


#### **2.3.2 - Reading and Decoding the Page**
From here on I work from the saved file, not the live site. The file is stored as bytes, and the page is in Russian, so the bytes must be decoded with the right encoding. The page declares `charset=UTF-8` (shown in the output), and the Russian text prints correctly, so UTF-8 is the right choice.

In [27]:
f = open("raw_html/donetsk_2024_03.html", "rb")  # open the saved file for reading bytes ("rb")
saved_html = f.read()  # read the whole file as bytes
f.close()  # close the file

text = saved_html.decode("utf-8")  # turn the bytes into text using UTF-8
print(text[:500])  # the start of the page: the charset line should say UTF-8, and the Russian should be readable

<!DOCTYPE html>
<html lang="ru">
<head>
<title>Архив погоды в Донецке за последнюю неделю, погода в Донецке за март 2024 года</title>
    <meta http-equiv="content-type" content="text/html; charset=UTF-8">
    <meta name="description" content="Архив фактической погоды в Донецке (атмосферное давление, температура воздуха, влажность воздуха, направление и скорость ветра, атмосферные явления, осадки) за март 2024 года.">
    <meta name="keywords" content="архив, погода, в Донецке">
    <meta name="


#### **2.3.3 - Finding the Tables**
I load the page into BeautifulSoup and list every table, with its class and number of rows, to find which tables hold the 3-hourly readings.

In [28]:
soup = BeautifulSoup(text, "html.parser")  # turn the HTML text into a searchable object
tables = soup.find_all("table")  # list of every <table> on the page
print(len(tables), "tables")  # how many there are

for i in range(len(tables)):  # go through the tables by position
    rows = tables[i].find_all("tr")  # all the rows in this table
    print(i, "| class:", tables[i].get("class"), "|", len(rows), "rows")  # table number, class and row count
    print(tables[i].get_text()[:200])  # first 200 characters of its text
    print("-----")  # separator between tables

2 tables
0 | class: None | 243 rows


Время  (UTC),	дата


00
1.03


03
1.03


06
1.03


09
1.03


12
1.03


15
1.03


18
1.03


21
1.03


00
2.03


03
2.03


06
2.03


09
2.03


12
2.03


15
2.03


00
3.03


03
3.03


06
3.03


09
3.03
-----
1 | class: None | 243 rows













Ветер (напр.,м/с)
Видим.
Явления
Облачность
Т(С)
Тd(С)
f(%)
Тe(С)
Тes(С)
Комфортность
P(гПа)
Po(гПа)
Тmin(С)
Tmax(С)
R(мм)
R24(мм)
S(см)


В
2
10 км

ясно
-0.8
-4.3
77
-4
-4
по сезону
103
-----


#### **Checking the Cells in One Row**
Before writing the parser, I print every cell of the first reading. `repr()` shows the exact text, including empty cells (`''`), to confirm each row has 18 cells and to see what format the values are in.

In [29]:
weather_rows = tables[1].find_all("tr")  # all rows of the weather table (bot=2 page)
first_row_cells = weather_rows[1].find_all("td")  # the cells of the first data row (row 0 is the header)
print(len(first_row_cells), "cells")  # expect 18
for cell in first_row_cells:  # go through each cell
    print(repr(cell.get_text()))  # show its exact text; repr() shows hidden spaces too

18 cells
'В'
'2'
'10 км'
''
'ясно'
'-0.8'
'-4.3'
'77'
'-4'
'-4'
'по сезону'
'1032.0'
'1004.0'
''
''
''
''
''


### **2.4 - The Bot Parameter**
The page's search form asks whether the visitor is a robot, an alien or a human, and my URL contains `bot=2`. I find the form's `bot` field to see what each value means.

In [30]:
bot_inputs = soup.find_all("input", {"name": "bot"})  # choice buttons named "bot", if the form uses them
for item in bot_inputs:  # go through each one
    print(item, "|", item.next_sibling)  # the button itself and the text label after it

bot_select = soup.find("select", {"name": "bot"})  # a drop-down list named "bot", if the form uses one instead
if bot_select is not None:  # only run this if a drop-down was found
    for option in bot_select.find_all("option"):  # each choice in the list
        print(option.get("value"), "|", option.get_text())  # the value it sends, and its label
        #0 = робот (robot)
        #1 = инопланетянин (alien)
        #2 = человек (human), ticked by default

<input name="bot" type="radio" value="0"/> | робот
<input name="bot" type="radio" value="1"/> | инопланетянин
<input checked="checked" name="bot" type="radio" value="2"/> | человек


#### **Testing bot=0**
To see what the bot answer changes, I request the same page (Donetsk, March 2024) answering "robot" (`bot=0`) and compare it with the saved `bot=2` page: size in bytes, number of rows, and the dates of the first and last reading. The test page is saved in a separate `test_html` folder, so it isn't counted as weather data.

In [31]:
# The same page, but answering "robot" (bot=0) instead of "human" (bot=2)
test_url = "http://www.pogodaiklimat.ru/weather.php?id=34519&bday=1&fday=31&amonth=3&ayear=2024&bot=0"
test_file = "test_html/donetsk_2024_03_bot0.html"  # kept out of raw_html so it isn't counted as weather data

os.makedirs("test_html", exist_ok=True)  # create the test folder if needed

if os.path.exists(test_file):  # already saved on an earlier run
    f = open(test_file, "rb")  # open the saved copy
    test_bytes = f.read()  # read its bytes
    f.close()  # close the file
else:  # first run only
    request = urllib.request.Request(test_url)  # build the request
    request.add_header("User-Agent", "Mozilla/5.0 (student project, TU Dublin)")  # identify the request
    test_bytes = opener.open(request).read()  # send it and read the raw bytes
    f = open(test_file, "wb")  # open a file for writing bytes
    f.write(test_bytes)  # save the test page
    f.close()  # close the file

print("bot=0:", len(test_bytes), "bytes | bot=2:", len(saved_html), "bytes")  # both in bytes, so they compare fairly

test_soup = BeautifulSoup(test_bytes.decode("utf-8"), "html.parser")  # make the bot=0 page searchable
test_rows = test_soup.find_all("table")[0].find_all("tr")  # rows of its time/date table
print(len(test_rows), "rows with bot=0 |", len(tables[0].find_all("tr")), "rows with bot=2")  # 53 vs 243 last time

first = test_rows[1].find_all("td")  # cells of the first reading (row 0 is the header)
last = test_rows[-1].find_all("td")  # cells of the last reading
print("bot=0 readings run from", first[1].get_text().strip(), first[0].get_text().strip(),
      "to", last[1].get_text().strip(), last[0].get_text().strip())  # date and UTC time of first and last reading

bot=0: 80856 bytes | bot=2: 228826 bytes
54 rows with bot=0 | 243 rows with bot=2
bot=0 readings run from 17.10 12 to 24.10 09


#### **Observations on the bot Test**
**What each value means:** The form's three answers are 0 = robot (робот), 1 = alien (инопланетянин) and 2 = human (человек), with human ticked by default.

**Less data:** Answering "robot" returned 54 rows (1 header + 53 readings) and 80,856 bytes, against 243 rows and 228,826 bytes for the "human" page.

**The wrong dates:** I asked for March 2024, but the robot page's readings run from 17 October 12:00 to 24 October 09:00 (UTC), the week up to the day I ran the test. So with `bot=0` the site ignores the requested dates and only shows the latest week. This also explains why the same test gave 53 rows on 23 October and 54 on 24 October: it is a moving one-week window, not part of March.

**What this means:** With `bot=0`, no historical data can be reached at all, so the scraper has to use `bot=2`.


### **2.5 - Parsing One Month**
The page has two tables with the same number of rows: `tables[0]` holds the time and date, and `tables[1]` holds the 18 weather values. Row `i` in one belongs to row `i` in the other, so I go through them together, skipping the header row (row 0), and put each reading into a list. I keep the values as text for now and clean them in a later step.

In [32]:
time_rows = tables[0].find_all("tr")  # rows of the time/date table
weather_rows = tables[1].find_all("tr")  # rows of the weather table

# English names for the 20 columns: 2 from the time table + 18 from the weather table
column_names = ["time_utc", "date", "wind_dir", "wind_speed", "visibility", "phenomena", "cloud",
                "temp", "dew_point", "humidity", "temp_effective", "temp_effective_sun", "comfort",
                "pressure_sea", "pressure_station", "temp_min", "temp_max", "precip", "precip_24h", "snow_depth"]

rows = []  # will hold one list of 20 values per reading

for i in range(1, len(time_rows)):  # start at 1 to skip the header row
    row = []  # the values for this reading
    for cell in time_rows[i].find_all("td"):  # the time and date cells
        row.append(cell.get_text().strip())  # add the text, with spaces trimmed off the ends
    for cell in weather_rows[i].find_all("td"):  # the 18 weather cells in the matching row
        row.append(cell.get_text().strip())  # add each one
    rows.append(row)  # store this reading

for row in rows:  # safety check: every reading should have exactly 20 values
    if len(row) != 20:  # if one doesn't
        print("Unexpected row:", row)  # show it so I can see what went wrong

march = pd.DataFrame(rows, columns=column_names)  # turn the list of readings into a DataFrame
print(march.shape)  # expect (242, 20)
march.head(10)  # look at the first 10 readings

(242, 20)


,time_utc,date,wind_dir,wind_speed,visibility,phenomena,cloud,temp,dew_point,humidity,temp_effective,temp_effective_sun,comfort,pressure_sea,pressure_station,temp_min,temp_max,precip,precip_24h,snow_depth
0,00,1.03,В,2,10 км,,ясно,-0.8,-4.3,77,-4,-4,по сезону,1032.0,1004.0,,,,,
1,03,1.03,штиль,0,4000 м,дымка,7 баллов,-1.3,-3.6,84,-2,-2,тепло,1030.7,1002.7,-1.3,,,,
2,06,1.03,В,4,10 км,,9 баллов,-1.2,-3.7,83,-7,-7,по сезону,1030.7,1002.7,-1.4,,,,
3,09,1.03,В,4,20 км,,9 баллов,+2.7,-1.1,76,-2,-1,по сезону,1030.3,1002.7,,,,,
4,12,1.03,В,2,20 км,,7 баллов,+7.9,-1.6,51,+6,+8,тепло,1029.8,1002.7,,,,,
5,15,1.03,В,2,20 км,,7 баллов,+6.1,-2.0,56,+4,+4,тепло,1028.5,1001.3,,+8.1,,,
6,18,1.03,В,2,10 км,,6 баллов,+2.5,-2.6,69,-0,-0,тепло,1028.9,1001.3,,,,,
7,21,1.03,В,2,10 км,,7 баллов,+0.8,-2.1,81,-3,-3,по сезону,1029.0,1001.3,,,,,
8,00,2.03,В,4,10 км,,7 баллов,+0.3,-2.1,84,-5,-5,по сезону,1027.7,1000.0,,,,,
9,03,2.03,В,4,10 км,,9 баллов,-0.2,-1.9,88,-6,-6,по сезону,1026.5,998.7,-0.2,,,,


#### **Checking for Modelled Values**
The site says rows in italics are modelled estimates rather than real observations. I count `<i>` tags and the word "italic" in the page's HTML to see whether any rows are marked.

In [33]:
print(text.count("<i>"), text.count("italic"))  # does the page mark any rows in italics (modelled values)?

0 0


#### **Checking Wind Gusts**
Gusts appear in curly brackets after the average speed (e.g. `8 {14}`). I list every reading with a gust, count them, and list every different wind speed value, to see how gusts are recorded and what steps the station reports in.

In [34]:
gust_count = 0  # counts readings that have a gust

for i in range(len(march)):  # go through every reading
    if "{" in march["wind_speed"][i]:  # a curly bracket means a gust was recorded
        gust_count = gust_count + 1  # count it
        print(march["date"][i], march["time_utc"][i], march["wind_speed"][i])  # show when, and the value

print(gust_count, "readings with a gust")  # total, out of 242
print(march["wind_speed"].unique())  # every different value, to see what steps the station reports in

2.03 09 4 {10}
2.03 12 6 {10}
2.03 15 6 {12}
3.03 00 8 {14}
3.03 03 8 {14}
3.03 06 8 {14}
3.03 09 8 {14}
3.03 12 8 {14}
3.03 15 8 {14}
3.03 18 6 {14}
3.03 21 6 {12}
4.03 00 6 {12}
4.03 03 6 {12}
4.03 06 6 {12}
4.03 12 6 {12}
4.03 15 6 {12}
4.03 18 4 {12}
6.03 18 4 {10}
10.03 15 2 {10}
10.03 18 2 {10}
13.03 00 6 {10}
13.03 03 6 {10}
13.03 06 8 {14}
13.03 09 8 {18}
13.03 12 10 {19}
13.03 15 10 {20}
13.03 18 12 {20}
13.03 21 10 {20}
14.03 00 8 {20}
14.03 03 8 {16}
14.03 06 8 {16}
14.03 09 6 {12}
14.03 12 6 {12}
14.03 15 6 {10}
14.03 18 6 {10}
14.03 21 4 {10}
15.03 00 6 {10}
15.03 03 6 {10}
15.03 06 6 {10}
15.03 09 6 {10}
15.03 12 6 {10}
20.03 09 4 {10}
20.03 12 4 {10}
20.03 15 6 {10}
20.03 18 4 {10}
24.03 09 6 {10}
24.03 12 6 {10}
24.03 15 6 {10}
24.03 18 4 {10}
28.03 12 6 {10}
28.03 15 6 {10}
28.03 18 4 {10}
29.03 15 6 {11}
29.03 18 6 {12}
30.03 06 6 {10}
55 readings with a gust
['2' '0' '4' '4 {10}' '6 {10}' '6 {12}' '8 {14}' '6 {14}' '4 {12}'
 '2 {10}' '8 {18}' '10 {19}' '10 {20}' '12 

#### **Observations on the Parsed Month (March 2024)**
**Broken rows:** The parsed table has 242 readings and 20 columns. Every reading lined up with its time and date, and no row had the wrong number of values.

**Missing readings:** A full month would have 248 readings (31 days × 8). This month has 242, so 6 are missing. For example, 2 March has no 18:00 or 21:00 reading. Days with missing readings need a rule when readings are combined into days (2.11).

**Date and time:** The date only contains day.month (e.g. `2.03` is 2 March), with no year, so the year is added from the file name. Times are in UTC, and Donetsk is UTC+3, so the 21:00 UTC reading is midnight local time and belongs to the next local day. This is fixed before readings are grouped into days.

**Values:** Every value is stored as text (e.g. `+2.7`, `10 км`, `ясно`, `штиль`) and needs converting to numbers. Visibility is given in both kilometres and metres, so it will all be converted to metres.

**Blank cells:** A blank doesn't always mean missing data. A blank `phenomena` cell means nothing was happening (no fog, rain or snow). `temp_min` is only filled in at early-morning readings and `temp_max` at 15:00 UTC, so they are blank the rest of the time. Precipitation is also only filled in at some readings.

**Wind:**
- Gusts are shown in curly brackets after the average speed, e.g. `8 {14}`. 55 of the 242 readings have a gust, and the smallest gust is 10 m/s, so a blank gust probably means the gust was below 10 m/s rather than missing. This is checked on the full data in 2.10.
- The windiest spells were 3–4 March and 13–15 March, with gusts up to 20 m/s.
- Average wind speeds are always even numbers (0, 2, 4 … 12), so the station reports them in steps of 2 m/s, while gusts are given to the nearest 1 m/s. That makes the gust the more precise wind measure.

**Modelled values:** No italic markers were found in this month's HTML, so none of these readings is marked as a modelled estimate.

**Columns not needed:** `comfort`, `temp_effective` and `temp_effective_sun` are worked out by the site from the other values and aren't needed for the research questions, so they are dropped in cleaning (2.10).

### **2.6 - Testing Before the Full Scrape**
Before scraping every month, I test the pages most likely to behave differently from March 2024:
- **Kharkiv, March 2024:** checks the second station works.
- **Donetsk, April 2024:** a 30-day month, requested with `fday=31`, to see how the site handles a day that doesn't exist.
- **Donetsk, August 2023:** a month with known gaps, to see what a partly empty page looks like.

I also check whether 2022 has any data, to decide where the scrape should start.

Each page is saved to `raw_html`, with a 3-second pause between requests. `time.sleep()` pauses the code, and `.zfill(2)` adds a leading zero to the month (4 → "04") so the files sort in order. The cell skips any page already saved, so re-running it sends no requests.

In [35]:
import time  # for pausing between requests

# Each test page: [station id, station name for the file, year, month]
test_pages = [[34300, "kharkiv", 2024, 3],
              [34519, "donetsk", 2024, 4],
              [34519, "donetsk", 2023, 8]]

for page in test_pages:  # go through each test page
    station_id = page[0]  # the station number
    station_name = page[1]  # the name used in the file name
    year = page[2]  # the year
    month = page[3]  # the month

    file_name = "raw_html/" + station_name + "_" + str(year) + "_" + str(month).zfill(2) + ".html"  # e.g. raw_html/donetsk_2024_04.html

    if os.path.exists(file_name):  # already saved on an earlier run
        print(file_name, "already saved, not downloaded again |", os.path.getsize(file_name), "bytes")
        continue  # go to the next test page without sending a request

    # build the URL for this station and month (days 1-31, as in the original test)
    url = ("http://www.pogodaiklimat.ru/weather.php?id=" + str(station_id)
           + "&bday=1&fday=31&amonth=" + str(month) + "&ayear=" + str(year) + "&bot=2")

    request = urllib.request.Request(url)  # build the request
    request.add_header("User-Agent", "Mozilla/5.0 (student project, TU Dublin)")  # identify the request
    raw_html = opener.open(request).read()  # send it and read the raw bytes

    f = open(file_name, "wb")  # open the file for writing bytes
    f.write(raw_html)  # save the page
    f.close()  # close the file

    print(file_name, len(raw_html), "bytes")  # what was saved and its size
    time.sleep(3)  # wait 3 seconds before the next request

raw_html/kharkiv_2024_03.html already saved, not downloaded again | 235823 bytes
raw_html/donetsk_2024_04.html already saved, not downloaded again | 220560 bytes
raw_html/donetsk_2023_08.html already saved, not downloaded again | 216644 bytes


#### **Checking the Test Pages**
For each saved test page, I count the rows in both tables and print the first and last reading, to check the page holds the month I asked for.

In [36]:
test_files = ["raw_html/kharkiv_2024_03.html",
              "raw_html/donetsk_2024_04.html",
              "raw_html/donetsk_2023_08.html"]

for file_name in test_files:  # go through each saved test page
    f = open(file_name, "rb")  # open the saved file for reading bytes
    page_text = f.read().decode("utf-8")  # read it and turn the bytes into text
    f.close()  # close the file

    page_soup = BeautifulSoup(page_text, "html.parser")  # make it searchable
    page_tables = page_soup.find_all("table")  # find all the tables
    print(file_name, "|", len(page_tables), "tables")  # file name and number of tables

    if len(page_tables) >= 2:  # only check rows if both tables are there
        time_rows = page_tables[0].find_all("tr")  # rows of the time/date table
        weather_rows = page_tables[1].find_all("tr")  # rows of the weather table
        print("  rows:", len(time_rows), "and", len(weather_rows))  # should be the same number
        if len(time_rows) > 1:  # only if there is at least one reading after the header
            first = time_rows[1].find_all("td")  # cells of the first reading
            last = time_rows[-1].find_all("td")  # cells of the last reading ([-1] = last item)
            print("  first:", first[1].get_text().strip(), first[0].get_text().strip())  # date and time of first reading
            print("  last: ", last[1].get_text().strip(), last[0].get_text().strip())  # date and time of last reading

raw_html/kharkiv_2024_03.html | 2 tables
  rows: 249 and 249
  first: 1.03 00
  last:  31.03 21
raw_html/donetsk_2024_04.html | 2 tables
  rows: 233 and 233
  first: 1.04 00
  last:  30.04 21
raw_html/donetsk_2023_08.html | 2 tables
  rows: 231 and 231
  first: 2.08 06
  last:  31.08 21


#### **Retesting April with the Correct Last Day**
The April page first came back with no readings (see *Observations on the Tests*). April has 30 days, so I test whether requesting day 31 caused this by asking for days 1–30 instead. The cell only re-downloads if the saved page is still the empty one.

In [37]:
file_name = "raw_html/donetsk_2024_04.html"  # the April page

# Empty pages are about 38 KB and full months over 200 KB, so 100 KB separates them safely
if os.path.getsize(file_name) < 100000:  # the saved page is an empty one
    url = "http://www.pogodaiklimat.ru/weather.php?id=34519&bday=1&fday=30&amonth=4&ayear=2024&bot=2"  # fday=30, April's real last day
    request = urllib.request.Request(url)  # build the request
    request.add_header("User-Agent", "Mozilla/5.0 (student project, TU Dublin)")  # identify the request
    raw_html = opener.open(request).read()  # send it and read the raw bytes
    f = open(file_name, "wb")  # overwrite the empty file
    f.write(raw_html)  # save the full page
    f.close()  # close the file
    print("Empty page replaced")
else:  # already the full month
    print("Saved page already has readings, not downloaded again")

f = open(file_name, "rb")  # open the saved page
april_soup = BeautifulSoup(f.read().decode("utf-8"), "html.parser")  # read, decode and make it searchable
f.close()  # close the file
april_rows = april_soup.find_all("table")[0].find_all("tr")  # rows of the time/date table
print(os.path.getsize(file_name), "bytes |", len(april_rows), "rows")  # first run gave 220,560 bytes, 233 rows

Saved page already has readings, not downloaded again
220560 bytes | 233 rows


#### **Checking the 2022 Coverage**
To decide where the scrape should start, I check whether the site has data for 2022. I request February and July 2022 for both stations (the start and end of the period that looked empty in earlier spot checks) and November 2022 for Donetsk, when its data seemed to begin. The pages are saved in `test_html`, so they aren't counted as weather data. *(This check was run after the main scrape, to confirm the start date I had chosen from the earlier spot checks.)*

In [38]:
# Pages to spot-check: [station id, name, year, month, last day of month]
check_pages = [[34519, "donetsk", 2022, 2, 28],
               [34300, "kharkiv", 2022, 2, 28],
               [34519, "donetsk", 2022, 7, 31],
               [34300, "kharkiv", 2022, 7, 31],
               [34519, "donetsk", 2022, 11, 30]]

os.makedirs("test_html", exist_ok=True)  # create the test folder if needed

for page in check_pages:  # go through each page to check
    file_name = "test_html/" + page[1] + "_" + str(page[2]) + "_" + str(page[3]).zfill(2) + ".html"  # e.g. test_html/donetsk_2022_02.html

    if not os.path.exists(file_name):  # only download if not saved on an earlier run
        url = ("http://www.pogodaiklimat.ru/weather.php?id=" + str(page[0])
               + "&bday=1&fday=" + str(page[4]) + "&amonth=" + str(page[3])
               + "&ayear=" + str(page[2]) + "&bot=2")  # URL with the month's real last day
        request = urllib.request.Request(url)  # build the request
        request.add_header("User-Agent", "Mozilla/5.0 (student project, TU Dublin)")  # identify the request
        raw = opener.open(request).read()  # send it and read the raw bytes
        f = open(file_name, "wb")  # open a file for writing bytes
        f.write(raw)  # save the page
        f.close()  # close the file
        time.sleep(3)  # pause before the next request

    f = open(file_name, "rb")  # open the saved page
    page_soup = BeautifulSoup(f.read().decode("utf-8"), "html.parser")  # read, decode, make searchable
    f.close()  # close the file
    readings = page_soup.find_all("table")[0].find_all("tr")[1:]  # every reading, skipping the header row

    if len(readings) == 0:  # an empty page
        print(file_name, "| 0 readings")
    else:  # show how many readings and which dates they cover
        first_date = readings[0].find_all("td")[1].get_text().strip()  # date of the first reading
        last_date = readings[-1].find_all("td")[1].get_text().strip()  # date of the last reading
        print(file_name, "|", len(readings), "readings | first:", first_date, "| last:", last_date)

test_html/donetsk_2022_02.html | 0 readings
test_html/kharkiv_2022_02.html | 0 readings
test_html/donetsk_2022_07.html | 0 readings
test_html/kharkiv_2022_07.html | 0 readings
test_html/donetsk_2022_11.html | 85 readings | first: 18.11 | last: 30.11


#### **Observations on the Tests**
*The download cells now read saved files, so re-running them shows the final pages (for example, April now has data). The results below are from the first run on 23 October 2026.*

**Kharkiv, March 2024:** 235,823 bytes and 248 readings (1–31 March, none missing), so the second station works the same way as Donetsk.

**Donetsk, April 2024 with `fday=31`:** only 38,549 bytes, and the tables held only the header row, with no readings. The page still loaded normally, so nothing signalled an error. Requested again with `fday=30` (April's real last day), it returned 220,560 bytes and 232 readings. So **`fday` must be the month's real last day**, and an empty page can be spotted by its size (about 38 KB, compared with over 200 KB for a full month).

**Donetsk, August 2023:** 216,644 bytes and 230 readings, starting on 2 August at 06:00. The station's data restarts partway through the month, after a gap.

**2022:** I expected the site to have weather data for 2022, because the weather carries on regardless of the invasion. But what the site records is the station's reports, not the weather itself, and both stations have no readings at all in February or July 2022. The whole of February is empty, including the 23 days before the invasion began on 24 February, so the invasion alone doesn't explain the gap. The pages give no reason for it, so I can't say what caused it. Donetsk only resumes on 18 November 2022, and even then about 18% of readings are missing.

**What this means for the full scrape:** each request uses the month's real last day (including 29 for February 2024), and the scrape starts in January 2023 for both stations: from then on the data is far more complete, and using the same start date at both stations keeps them comparable.

### **2.7 - Scraping Every Month**
The tests in 2.6 showed that requesting a day that doesn't exist returns an empty page, so each request uses the month's real last day. February has 28 days, except in 2024, a leap year, when it has 29.

The loop goes through both stations and every month from January 2023 to October 2026 (92 pages). Each page is saved to `raw_html` before anything else is done with it. `os.path.exists()` checks whether a file is already saved; if it is (such as the test pages), the loop skips it, so no page is downloaded twice, and the loop can be re-run without repeating requests. There is a 3-second pause between requests.

When first run, the loop downloaded 88 pages and skipped the 4 already saved. The output below is from a later re-run, which confirms all 92 files exist and sends no requests.

In [39]:
# The two stations: [station id, name used in file names]
stations = [[34519, "donetsk"],
            [34300, "kharkiv"]]

years = [2023, 2024, 2025, 2026]  # years to scrape

days_in_month = [31, 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31]  # Jan to Dec; position 0 = January

downloaded = 0  # counts new pages fetched
skipped = 0  # counts pages already saved

for station in stations:  # go through each station
    station_id = station[0]  # the station number
    station_name = station[1]  # the name used in the file name

    for year in years:  # go through each year
        for month in range(1, 13):  # months 1 to 12
            if year == 2026 and month > 10:  # Kaggle data ends in October 2026
                continue  # skip November and December 2026

            last_day = days_in_month[month - 1]  # month 1 is position 0 in the list
            if year == 2024 and month == 2:  # 2024 is a leap year
                last_day = 29  # February 2024 has 29 days

            file_name = "raw_html/" + station_name + "_" + str(year) + "_" + str(month).zfill(2) + ".html"  # e.g. raw_html/kharkiv_2025_06.html

            if os.path.exists(file_name):  # already saved, e.g. one of the test pages
                skipped = skipped + 1  # count it
                continue  # move on without sending a request

            url = ("http://www.pogodaiklimat.ru/weather.php?id=" + str(station_id)
                   + "&bday=1&fday=" + str(last_day) + "&amonth=" + str(month)
                   + "&ayear=" + str(year) + "&bot=2")  # URL with this month's real last day

            request = urllib.request.Request(url)  # build the request
            request.add_header("User-Agent", "Mozilla/5.0 (student project, TU Dublin)")  # identify the request
            response = opener.open(request)  # send it
            raw_html = response.read()  # the page as raw bytes

            f = open(file_name, "wb")  # open the file for writing bytes
            f.write(raw_html)  # save the page
            f.close()  # close the file

            downloaded = downloaded + 1  # count it
            print(file_name, len(raw_html), "bytes")  # show progress
            time.sleep(3)  # pause 3 seconds before the next request

print("Downloaded:", downloaded, "| Skipped (already saved):", skipped)  # first run: 88 downloaded, 4 skipped

Downloaded: 0 | Skipped (already saved): 92


#### **Re-fetching October 2026**
Both October 2026 pages came back empty (about 38 KB, the same size as the empty June–July 2023 pages). Day 31 is in the future, and like the 30-day months, the site seems to return nothing for a range it can't fill. I re-fetch October with day 20 as the last day, which covers the Kaggle data (to 10 October).

In [40]:
for station in stations:  # both stations, from the scraping cell
    file_name = "raw_html/" + station[1] + "_2026_10.html"  # the October 2026 page for this station

    if os.path.getsize(file_name) > 100000:  # already a page with readings (empty pages are about 38 KB)
        print(file_name, "already has readings, not downloaded again |", os.path.getsize(file_name), "bytes")
        continue  # go to the next station without sending a request

    url = ("http://www.pogodaiklimat.ru/weather.php?id=" + str(station[0])
           + "&bday=1&fday=20&amonth=10&ayear=2026&bot=2")  # October 2026, days 1-20

    request = urllib.request.Request(url)  # build the request
    request.add_header("User-Agent", "Mozilla/5.0 (student project, TU Dublin)")  # identify the request
    raw_html = opener.open(request).read()  # send it and read the raw bytes

    f = open(file_name, "wb")  # replace the empty file
    f.write(raw_html)  # save
    f.close()  # close

    print(file_name, len(raw_html), "bytes")  # first run gave 163,485 and 166,835
    time.sleep(3)  # pause before the next request

raw_html/donetsk_2026_10.html already has readings, not downloaded again | 163485 bytes
raw_html/kharkiv_2026_10.html already has readings, not downloaded again | 166835 bytes


### **2.8 - Checking Coverage**
Before parsing every page, I check how complete the scraped data is. For each saved page I count the readings (rows minus the header) and the number of different days, to find months with gaps. `os.listdir()` lists the files in a folder, and `set()` keeps each date only once, so its length is the number of different days.

In [41]:
coverage_rows = []  # one entry per file

for file_name in sorted(os.listdir("raw_html")):  # every saved page, in name order
    f = open("raw_html/" + file_name, "rb")  # open the saved page
    page_soup = BeautifulSoup(f.read().decode("utf-8"), "html.parser")  # read, decode and make it searchable
    f.close()  # close the file

    time_rows = page_soup.find_all("table")[0].find_all("tr")  # rows of the time/date table
    dates = []  # the date of every reading
    for row in time_rows[1:]:  # skip the header row
        dates.append(row.find_all("td")[1].get_text().strip())  # second cell is the date

    parts = file_name.replace(".html", "").split("_")  # e.g. "donetsk_2024_04" -> ["donetsk", "2024", "04"]
    coverage_rows.append([parts[0], int(parts[1]), int(parts[2]), len(dates), len(set(dates))])  # station, year, month, readings, days

coverage = pd.DataFrame(coverage_rows, columns=["station", "year", "month", "readings", "days"])  # turn into a table
print(coverage.shape)  # expect (92, 5)
coverage[coverage["readings"] < 200]  # show only the months with fewer than 200 readings

(92, 5)


,station,year,month,readings,days
5,donetsk,2023,6,0,0
6,donetsk,2023,7,0,0
45,donetsk,2026,10,157,20
91,kharkiv,2026,10,160,20


#### **Missing Days and Readings per Month**
The coverage check above only finds months with very few readings. Here I work out how many days and readings each month should have (8 readings per day) and compare them with what was scraped. October 2026 was requested up to the 20th, so it should have 20 days. `.groupby("station")` adds up the totals for each station.

In [42]:
expected = []  # the number of days each month should have

for i in range(len(coverage)):  # go through every month in the table
    year = coverage["year"][i]  # this month's year
    month = coverage["month"][i]  # this month's number
    days = days_in_month[month - 1]  # normal length, from the scraping cell
    if year == 2024 and month == 2:  # leap year
        days = 29
    if year == 2026 and month == 10:  # only requested up to the 20th
        days = 20
    expected.append(days)  # store it

coverage["expected_days"] = expected  # add as a new column
coverage["missing_days"] = coverage["expected_days"] - coverage["days"]  # whole days with no readings
coverage["missing_readings"] = coverage["expected_days"] * 8 - coverage["readings"]  # readings missing in total

print(coverage.groupby("station")[["readings", "missing_days", "missing_readings"]].sum())  # totals for each station
coverage[coverage["missing_days"] > 0]  # months missing at least one whole day

         readings  missing_days  missing_readings
station                                          
donetsk     10351            63               761
kharkiv     11123             0               -11


,station,year,month,readings,days,expected_days,missing_days,missing_readings
5,donetsk,2023,6,0,0,30,30,240
6,donetsk,2023,7,0,0,31,31,248
7,donetsk,2023,8,230,30,31,1,18
13,donetsk,2024,2,210,28,29,1,22


#### **Investigating the Odd Months**
Kharkiv has 11 more readings than 8 per day allows, so I check which months have extra readings, which times they are at, and whether any date and time appears twice.

In [43]:
extra = coverage[coverage["missing_readings"] < 0]  # months with more readings than expected
print(extra[["station", "year", "month", "readings", "expected_days"]])  # show which months

for i in extra.index:  # go through each of those months
    file_name = ("raw_html/" + coverage["station"][i] + "_" + str(coverage["year"][i])
                 + "_" + str(coverage["month"][i]).zfill(2) + ".html")  # rebuild the file name
    f = open(file_name, "rb")  # open the saved page
    page_soup = BeautifulSoup(f.read().decode("utf-8"), "html.parser")  # read and make it searchable
    f.close()  # close the file

    times = []  # time of every reading
    pairs = []  # date and time of every reading together
    for row in page_soup.find_all("table")[0].find_all("tr")[1:]:  # every reading, skipping the header
        cells = row.find_all("td")  # the time and date cells
        times.append(cells[0].get_text().strip())  # the time
        pairs.append(cells[1].get_text().strip() + " " + cells[0].get_text().strip())  # e.g. "14.01 03"

    print(file_name)
    print(pd.Series(times).value_counts().sort_index())  # how many readings at each time; normal times are 00, 03 ... 21
    print("Duplicate date and time pairs:", len(pairs) - len(set(pairs)))  # 0 means no reading appears twice

    station  year  month  readings  expected_days
46  kharkiv  2023      1       250             31
50  kharkiv  2023      5       259             31
raw_html/kharkiv_2023_01.html
00    32
03    32
06    31
09    31
12    31
15    31
18    31
21    31
Name: count, dtype: int64
Duplicate date and time pairs: 2
raw_html/kharkiv_2023_05.html
00    31
03    31
06    31
09    33
12    37
15    32
18    31
21    33
Name: count, dtype: int64
Duplicate date and time pairs: 11


#### **Finding the Missing Day in February 2024**
The coverage table shows Donetsk is missing one whole day in February 2024. I list every date on the page and print any day from 1 to 29 that never appears.

In [44]:
f = open("raw_html/donetsk_2024_02.html", "rb")  # Donetsk, February 2024
page_soup = BeautifulSoup(f.read().decode("utf-8"), "html.parser")  # read and make it searchable
f.close()  # close the file

feb_dates = []  # dates found on the page
for row in page_soup.find_all("table")[0].find_all("tr")[1:]:  # every reading, skipping the header
    feb_dates.append(row.find_all("td")[1].get_text().strip())  # the date, e.g. "14.02"

for day in range(1, 30):  # days 1 to 29
    if str(day) + ".02" not in feb_dates:  # if this day never appears
        print("Missing:", str(day) + ".02")  # show it

Missing: 13.02


#### **Observations on the Scrape and Coverage**
**Coverage:** 92 pages were saved (46 months × 2 stations, January 2023 to 20 October 2026). At 8 readings a day, each station should have 1,389 days and 11,112 readings.

| Station | Readings | Missing days | Missing readings | Duplicates |
|---|---|---|---|---|
| Donetsk | 10,351 | 63 | 761 (about 7%) | not yet checked (see 2.10) |
| Kharkiv | 11,123 | 0 | 2 | 13 found so far |

**Donetsk:**
- June and July 2023 have no readings at all (61 days). This is the main gap in the weather data. June was requested with its correct last day (30), and other 30-day months requested the same way returned full data, so this is a real gap in the station's data and not a request error.
- Two other single days are missing: 1 August 2023 (the data restarts on the 2nd) and 13 February 2024. The pages give no reason.
- Outside the June–July gap, only about 2.6% of readings are missing, so the station is otherwise fairly complete.

**Kharkiv:**
- No whole days are missing, but the station has 11 more readings than 8 per day allows. Checking the two months responsible showed these are duplicates, not extra reports: every reading is at one of the normal times (00, 03 … 21), but some appear twice, 2 in January 2023 and 11 in May 2023.
- That makes 13 duplicates, while the total was only 11 too many, so Kharkiv is also missing 2 readings elsewhere. Adding up missing and extra readings hid both problems.
- This check only finds duplicates in months that have *more* readings than expected. A month with one duplicate and one missing reading would look normal, and Donetsk wasn't checked for duplicates at all. So 2.10 checks every reading at both stations.

**What this means for the next steps:**
- Duplicate readings must be removed before readings are combined into days, or they would be counted twice. 2.10 also checks whether the two copies of each duplicate have the same values.
- Days with no readings (the Donetsk gap, 1 August 2023 and 13 February 2024) will have no weather data after the merge. I leave them empty rather than filling them from Kharkiv, and use Kharkiv as a second station to check whether results hold.

### **2.9 - Parsing Every Page**
I run the same parser as in 2.5 on all 92 saved pages. The dates on the page have no year, so for each reading I also add the station, year and month, taken from the file name (e.g. `donetsk_2024_03.html`). All values stay as text for now; they are cleaned in 2.10. Pages with no readings, such as Donetsk in June and July 2023, add no rows.

Because each reading is matched to its time and date by row position, the code also checks that each page's two tables have the same number of rows, and that every reading ends up with exactly 23 values (3 from the file name + 2 time and date + 18 weather).

In [45]:
all_rows = []  # will hold one list of 23 values per reading, from every page

for file_name in sorted(os.listdir("raw_html")):  # every saved page, in name order
    parts = file_name.replace(".html", "").split("_")  # "donetsk_2024_03.html" -> ["donetsk", "2024", "03"]
    station = parts[0]  # station name
    year = int(parts[1])  # year, turned from text into a number
    month = int(parts[2])  # month, turned from text into a number

    f = open("raw_html/" + file_name, "rb")  # open the saved page for reading bytes
    page_soup = BeautifulSoup(f.read().decode("utf-8"), "html.parser")  # read, decode and make it searchable
    f.close()  # close the file

    page_tables = page_soup.find_all("table")  # the page's two tables
    time_rows = page_tables[0].find_all("tr")  # rows of the time/date table
    weather_rows = page_tables[1].find_all("tr")  # rows of the weather table

    if len(time_rows) != len(weather_rows):  # readings are matched by position, so the counts must agree
        print("Row counts differ:", file_name, len(time_rows), len(weather_rows))  # show any page where they don't

    for i in range(1, len(time_rows)):  # start at 1 to skip the header row
        row = [station, year, month]  # start each reading with the file details
        for cell in time_rows[i].find_all("td"):  # the time and date cells
            row.append(cell.get_text().strip())  # add the text, trimmed
        for cell in weather_rows[i].find_all("td"):  # the 18 weather cells in the matching row
            row.append(cell.get_text().strip())  # add each one
        all_rows.append(row)  # store this reading

broken = 0  # counts readings without exactly 23 values
for row in all_rows:  # check every reading
    if len(row) != 23:  # 3 file details + 2 time/date + 18 weather
        broken = broken + 1  # count it
        print("Unexpected row:", row)  # show it so I can see what went wrong
print(broken, "broken rows")  # should be 0

weather_raw = pd.DataFrame(all_rows, columns=["station", "year", "month"] + column_names)  # one table of every reading
print(weather_raw.shape)  # (rows, columns)

0 broken rows
(21474, 23)


#### **Checking the Parsed Data**
The number of readings per station should match the coverage check in 2.8, and the first rows should show the new station, year and month columns.

In [46]:
print(weather_raw["station"].value_counts())  # readings per station; compare with 2.8
weather_raw.head()  # the first 5 readings, with station, year and month at the start

station
kharkiv    11123
donetsk    10351
Name: count, dtype: int64


,station,year,month,time_utc,date,wind_dir,wind_speed,visibility,phenomena,cloud,...,temp_effective,temp_effective_sun,comfort,pressure_sea,pressure_station,temp_min,temp_max,precip,precip_24h,snow_depth
0,donetsk,2023,1,00,1.01,штиль,0,1000 м,,н/о,...,,,,1032.4,1004.0,,,,,
1,donetsk,2023,1,03,1.01,ЮЗ,4,1000 м,,9 баллов,...,-5,-5,по сезону,1031.1,1002.7,+0.3,,,,
2,donetsk,2023,1,06,1.01,ЮЗ,2,10 км,,9 баллов,...,-2,-2,тепло,1031.1,1002.7,+0.3,,,,
3,donetsk,2023,1,09,1.01,ЮЗ,4,10 км,,9 баллов,...,+3,+3,тепло,1029.0,1001.3,,,,,
4,donetsk,2023,1,12,1.01,ЮЗ,6 {10},10 км,,9 баллов,...,+4,+4,тепло,1027.6,1000.0,,,,,


#### **Observations on the Parsed Data**
**Prediction:** I expected 23 columns (3 from the file name and 20 from the page), and I expected Donetsk June and July 2023 to add nothing, because those pages had no readings.

**Result:** `weather_raw` has 21,474 rows and 23 columns. 21,474 is exactly 10,351 + 11,123, the reading counts from the coverage check in 2.8, so every reading on every page was parsed once, with none lost or added. No page had tables with different row counts, and no reading had the wrong number of values (0 broken rows), so matching readings to their time and date by row position worked on all 92 pages.

**June and July 2023:** The empty pages created no rows at all, so there is nothing to drop. The gap will show up after the merge as days with no weather data.

**Still to fix:** Kharkiv's count still includes the 13 duplicates found in 2.8, and every value is still text. Both are dealt with in 2.10.

**A first look at the rows:** The first reading (Donetsk, 1 January 2023, 00:00) has visibility of 1,000 m and cloud `н/о` (not determined), but a blank `phenomena` cell. The site's column notes say visibility of 1 km or less usually means fog, so a blank `phenomena` cell may not always mean clear weather. I check how often this happens in 2.10 before deciding how to flag fog.

### **2.10 - Cleaning**
The parsed values are all text, and the data contains duplicate readings. This section has three steps:

**1.** Remove duplicate readings.

**2.** Drop the columns not needed for the research questions.

**3.** Convert the text values to numbers.

#### **2.10.1 - Removing Duplicate Readings**
A reading is identified by its station, year, date and time. Two rows with the same four values are the same reading recorded twice. `.duplicated(subset=key)` marks every repeat of an earlier row with the same key. `.duplicated()` with no subset marks rows that repeat an earlier row in *every* column. If the two counts are equal, every duplicate is an exact copy. If the second is smaller, some copies have different values, and I need to decide which to keep.

In [47]:
key = ["station", "year", "date", "time_utc"]  # the columns that identify one reading

same_key = weather_raw.duplicated(subset=key)  # True for every repeat of an earlier reading with the same key
same_everything = weather_raw.duplicated()  # True only if the repeat matches in every column

print(same_key.sum(), "duplicate readings (same station, year, date and time)")  # how many repeats
print(same_everything.sum(), "of them identical in every column")  # how many are exact copies

dupes = weather_raw[weather_raw.duplicated(subset=key, keep=False)]  # keep=False marks both copies, not just the repeat
print(dupes.groupby("station").size())  # how many rows are involved at each station
dupes.sort_values(key).head(10)  # first few pairs side by side, to compare the copies

13 duplicate readings (same station, year, date and time)
5 of them identical in every column
station
kharkiv    21
dtype: int64


,station,year,month,time_utc,date,wind_dir,wind_speed,visibility,phenomena,cloud,...,temp_effective,temp_effective_sun,comfort,pressure_sea,pressure_station,temp_min,temp_max,precip,precip_24h,snow_depth
10351,kharkiv,2023,1,00,1.01,Ю,1,10 км,,10/0[Ac und],...,0,0,тепло,1024.6,1006.5,,,,,
10352,kharkiv,2023,1,00,1.01,Ю,1,н/о,,10 баллов,...,0,0,тепло,1026.2,1006.7,,,,,
10353,kharkiv,2023,1,03,1.01,Ю,2,н/о,в посл. час дождь,10 баллов,...,0,0,тепло,1023.3,1004.0,,,,,
10354,kharkiv,2023,1,03,1.01,Ю,2,10 км,в посл. час дождь,10/10 300 м[Sc],...,0,0,тепло,1022.5,1004.5,,,,,
11315,kharkiv,2023,5,09,1.05,СЗ,2,10 км,слаб. дождь,10/10 300 м[Sc],...,+7,+7,холодно,1014.0,996.5,,,,,
11316,kharkiv,2023,5,09,1.05,СЗ,2,н/о,слаб. дождь,10 баллов,...,+7,+7,холодно,1014.8,996.0,,,,,
11320,kharkiv,2023,5,21,1.05,штиль,0,10 км,,ясно,...,+5,+5,прохладно,1021.1,1003.3,,,,,
11321,kharkiv,2023,5,21,1.05,штиль,0,н/о,,ясно,...,+5,+5,прохладно,1023.1,1004.0,,,,,
11357,kharkiv,2023,5,09,6.05,СЗ,2,10 км,,"6/6 1000 м[Cu med, cong]",...,+15,+15,тепло,1022.5,1004.0,,,,,
11358,kharkiv,2023,5,09,6.05,СЗ,2,н/о,,6 баллов,...,+15,+15,тепло,1022.5,1004.0,,,,,


#### **Comparing the Duplicate Copies**
Some duplicate copies have different values. I show all of them side by side, with the columns that differ, to see whether the differences follow a pattern before deciding which copy to keep.

In [48]:
dupes = weather_raw[weather_raw.duplicated(subset=key, keep=False)]  # every row involved in a duplicate
dupes.sort_values(key)[key + ["visibility", "cloud", "pressure_sea", "pressure_station"]]  # all 21 rows, key columns plus the ones that differ

,station,year,date,time_utc,visibility,cloud,pressure_sea,pressure_station
10351,kharkiv,2023,1.01,00,10 км,10/0[Ac und],1024.6,1006.5
10352,kharkiv,2023,1.01,00,н/о,10 баллов,1026.2,1006.7
10353,kharkiv,2023,1.01,03,н/о,10 баллов,1023.3,1004.0
10354,kharkiv,2023,1.01,03,10 км,10/10 300 м[Sc],1022.5,1004.5
11315,kharkiv,2023,1.05,09,10 км,10/10 300 м[Sc],1014.0,996.5
11316,kharkiv,2023,1.05,09,н/о,10 баллов,1014.8,996.0
11320,kharkiv,2023,1.05,21,10 км,ясно,1021.1,1003.3
11321,kharkiv,2023,1.05,21,н/о,ясно,1023.1,1004.0
11357,kharkiv,2023,6.05,09,10 км,"6/6 1000 м[Cu med, cong]",1022.5,1004.0
11358,kharkiv,2023,6.05,09,н/о,6 баллов,1022.5,1004.0


#### **Keeping One Copy of Each Reading**
Where the copies differ, one has a measured visibility and detailed cloud, and the other has visibility `н/о` and cloud amount only. Visibility is needed for the fog hypothesis, so I keep the copy that has it. I add a helper column marking rows where visibility is `н/о`, sort so that within each reading the copy with visibility comes first, and keep only the first copy. `.drop_duplicates(subset=key, keep="first")` keeps the first row of each reading and removes the rest.

In [49]:
weather = weather_raw.copy()  # work on a copy, so weather_raw stays as parsed
weather["no_visibility"] = weather["visibility"] == "н/о"  # True where visibility wasn't recorded

weather = weather.sort_values(key + ["no_visibility"])  # within each reading, False (has visibility) sorts before True
weather = weather.drop_duplicates(subset=key, keep="first")  # keep the first copy of each reading
weather = weather.drop(columns=["no_visibility"])  # remove the helper column
weather = weather.sort_index()  # put the rows back in their original order

print(weather.shape)  # expect 21,474 - 13 = 21,461 rows
print(weather.duplicated(subset=key).sum(), "duplicates left")  # should be 0

(21461, 23)
0 duplicates left


#### **Observations on the Duplicates**
**Where they are:** All duplicates are at Kharkiv, on three days: 1 January, 1 May and 6 May 2023. The full check found none at Donetsk. The 21 duplicate rows are 8 different readings, so some readings appear more than twice.

**Two kinds of duplicate:**
- **Exact copies:** the 6 May 12:00 reading appears six times with identical values (rows 11359 and 11361–11365). These are the 5 repeats that are identical in every column.
- **Different reports for the same time:** every one of the 8 readings also has one extra copy where visibility is `н/о` (not determined) and the cloud is given only as an amount (e.g. `10 баллов`), instead of the full copy's amount, cloud base height and cloud type (e.g. `10/10 300 м[Sc]`). The pressures also differ slightly between the two. The station pressure, which is measured, differs by up to 0.7 hPa, while the sea-level pressure, which is calculated from it, differs by up to 2 hPa. This suggests two separate reports for the same time, one full and one reduced, but the page doesn't say where each comes from.

**Which copy is kept:** Visibility is needed for the fog hypothesis, so I keep the copy with a measured visibility. Sorting first matters: for 1 January 03:00 the `н/о` copy comes first on the page, so simply keeping the first copy would have kept the worse one.

**Effect:** 13 rows were removed, leaving 21,461 readings. That is 0.06% of the data, so this choice hardly changes any daily values; what matters is that the rule is stated and applied consistently.

#### **2.10.2 - Dropping Columns Not Needed**
I keep only the columns needed to identify each reading or to test a hypothesis from 2.1: wind speed and gusts, visibility, weather phenomena, cloud, temperature (current, minimum and maximum), precipitation and snow depth. I also keep humidity for now: fog only forms in nearly saturated air, so humidity may help where visibility wasn't recorded. I decide this after checking how often that happens (2.10.3).

I drop wind direction (direction only matters relative to the way a drone or shell is travelling, which the loss data doesn't record), dew point (it carries the same information as humidity), the three comfort columns the site calculates for people (see 2.5), and both air pressures (no hypothesis involves pressure).

In [50]:
drop_columns = ["wind_dir", "dew_point", "temp_effective", "temp_effective_sun",
                "comfort", "pressure_sea", "pressure_station"]  # columns not needed for the research questions

weather = weather.drop(columns=drop_columns)  # remove them
print(weather.shape)  # expect (21461, 16): 23 columns - 7 dropped
print(list(weather.columns))  # the columns left

(21461, 16)
['station', 'year', 'month', 'time_utc', 'date', 'wind_speed', 'visibility', 'phenomena', 'cloud', 'temp', 'humidity', 'temp_min', 'temp_max', 'precip', 'precip_24h', 'snow_depth']


#### **2.10.3 - Converting Text to Numbers**
Before writing conversion rules, I check every value in the full data, not just March. For each column, I count the blanks and list every value that isn't a plain number (for example `10 км`, `н/о` or `6 {12}`). I also check that `time_utc` only contains the 8 normal reading times. These values decide the conversion rules and the glossary below.

In [51]:
print(weather["time_utc"].unique())  # should only be 00, 03, 06 ... 21

check_columns = ["wind_speed", "visibility", "phenomena", "cloud", "temp", "humidity",
                 "temp_min", "temp_max", "precip", "precip_24h", "snow_depth"]  # columns to inspect

for col in check_columns:  # go through each column
    values = weather[col].unique()  # every different value in this column
    not_plain = []  # values that aren't just a number
    for value in values:  # go through each different value
        stripped = value.replace("+", "").replace("-", "").replace(".", "")  # remove signs and the decimal point
        if value != "" and not stripped.isdigit():  # not blank, and not only digits once signs are removed
            not_plain.append(value)  # keep it to look at
    blanks = (weather[col] == "").sum()  # how many readings are blank in this column
    print(col, "|", blanks, "blank |", len(values), "different values |", len(not_plain), "not plain numbers")
    print(not_plain[:50])  # up to 50 examples
    print("-----")

['00' '03' '06' '09' '12' '15' '18' '21']
wind_speed | 3 blank | 169 different values | 154 not plain numbers
['6 {10}', '4 {10}', '6 {14}', '6 {12}', '4 {12}', '8 {16}', '8 {18}', '10 {18}', '8 {14}', '6 {16}', '2 {12}', '2 {10}', '6-10 {10}', '10-15 {15}', '6-12 {15}', '6-10 {12}', '6-12 {12}', '6-12 {14}', '10-16 {16}', '8-16 {16}', '6-12 {16}', '10 {15}', '12-20 {20}', '16-24 {24}', '12-20 {24}', '6-12 {24}', '10-16 {20}', '10-16 {18}', '8-16 {18}', '6-21 {14}', '11-21 {21}', '10-18 {18}', '12 {16}', '{21}', '8 {12}', '6 {11}', '10 {14}', '4 {14}', '10 {19}', '10 {20}', '12 {20}', '8 {20}', '8 {15}', '6 {24}', '6 {15}', '8 {10}', '4 {16}', '8 {11}', '10 {16}', '2 {14}']
-----
visibility | 0 blank | 10 different values | 10 not plain numbers
['1000 м', '10 км', '4000 м', '2000 м', 'н/о', '500 м', '200 м', '50 м', '20 км', '50 км']
-----
phenomena | 15897 blank | 324 different values | 323 not plain numbers
['слаб. дождь', '{дождь}', 'дымка', 'дождь', 'слаб. дождь {морось}', '{морось

#### **Glossary of Russian Terms and Conversion Rules**
| Russian | English | Column | Converted to |
|---|---|---|---|
| `м` / `км` | metres / kilometres | visibility | metres (km × 1,000) |
| `н/о` (не определено) | not determined | visibility, cloud | empty |
| `ясно` | clear sky | cloud | 0 |
| `балл`, `балла`, `баллов` | tenths of the sky covered | cloud | the number (0–10) |
| `10/10 300 м[Sc]` | total / low cloud, cloud base height, cloud type | cloud | the first number (total cloud) |
| `?/?` | unknown | cloud | empty |
| `6 {10}` | average speed, with gust in `{ }` (m/s) | wind_speed | `wind_avg` = 6, `wind_gust` = 10 |
| `6-10` | range of average speeds | wind_speed | `wind_avg` = midpoint (8) |
| `{21}` | gust only | wind_speed | `wind_avg` empty, `wind_gust` = 21 |
| `туман` | fog | phenomena | `fog` = 1 |
| `туман, небо не видно` | fog, sky not visible | phenomena | `fog` = 1 |
| `дымка` | haze (visibility 1–10 km) | phenomena | not counted as fog |
| `дождь` / `ливневой дождь` | rain / rain shower | phenomena | `rain` = 1 |
| `морось` | drizzle | phenomena | `rain` = 1 |
| `переохл. дождь / морось / осадки` | freezing rain / drizzle / precipitation | phenomena | `rain` = 1 |
| `снег` / `ливневой снег` | snow / snow shower | phenomena | `snow` = 1 |
| `метель` | snowstorm (blowing snow) | phenomena | `snow` = 1 |
| `гололёд`, `диам. гололёда N мм` | glaze ice, ice deposit N mm across | phenomena | not used |
| `слаб.` | light (weak) | phenomena | ignored: any amount counts |
| `в посл. час` | in the last hour | phenomena | counted |
| `{ … }` in phenomena | seen 1–3 hours before the reading | phenomena | counted |
| `<0.5` | less than 0.5 cm | snow_depth | 0.5 |

Weather in `{ }` or "in the last hour" still happened that day, so it counts towards the day's flags. Haze (`дымка`) is not counted as fog: the site defines it as visibility of 1–10 km, while fog is 1 km or less.

#### **Converting the Number Columns**
`temp`, `humidity`, `temp_min`, `temp_max` and `precip` are already numbers stored as text, so `float()` converts them directly, and blanks stay empty. In `snow_depth`, `<0.5` (less than half a centimetre) becomes 0.5. `precip_24h` is blank in every reading, so it is dropped.

In [56]:
number_columns = ["temp", "humidity", "temp_min", "temp_max", "precip"]  # columns that only need text -> number

for col in number_columns:  # go through each column
    converted = []  # the new numeric values
    for value in weather[col]:  # go through every reading
        if value == "":  # blank
            converted.append(None)  # stays empty (NaN)
        else:
            converted.append(float(value))  # e.g. "+2.7" -> 2.7
    weather[col] = converted  # replace the text column with numbers

snow = []  # the new snow depth values
for value in weather["snow_depth"]:  # go through every reading
    if value == "":  # blank
        snow.append(None)  # stays empty
    elif value == "<0.5":  # less than half a centimetre
        snow.append(0.5)  # recorded as 0.5
    else:
        snow.append(float(value))  # e.g. "3" -> 3.0
weather["snow_depth"] = snow  # replace with numbers

if "precip_24h" in weather.columns:  # only drop it if it's still there
    weather = weather.drop(columns=["precip_24h"])  # blank in every reading

#### **Splitting Wind into Average and Gust**
`wind_speed` holds two values. The gust, if there is one, is the number inside `{ }`. The rest is the average speed: one number, a range such as `6-10` (I take the midpoint), or nothing. I put them in two new columns, `wind_avg` and `wind_gust`, and drop the original.

In [58]:
wind_avg = []  # average speeds
wind_gust = []  # gusts (empty where none recorded)
ranges = 0  # counts readings where the average is a range

for value in weather["wind_speed"]:  # go through every reading
    gust = None  # no gust unless one is found
    speed_text = value  # the part that holds the average speed
    if "{" in value:  # a gust is recorded
        gust = float(value[value.find("{") + 1 : value.find("}")])  # the number between { and }
        speed_text = value[:value.find("{")]  # everything before the {
    speed_text = speed_text.strip()  # remove spaces

    if speed_text == "":  # no average speed (blank, or gust only)
        wind_avg.append(None)
    elif "-" in speed_text:  # a range such as "6-10"
        low = float(speed_text.split("-")[0])  # number before the dash
        high = float(speed_text.split("-")[1])  # number after the dash
        wind_avg.append((low + high) / 2)  # midpoint
        ranges = ranges + 1  # count it
    else:
        wind_avg.append(float(speed_text))  # a single number
    wind_gust.append(gust)  # store the gust (or None)

weather["wind_avg"] = wind_avg  # new column
weather["wind_gust"] = wind_gust  # new column
weather = weather.drop(columns=["wind_speed"])  # the original text is no longer needed
print(ranges, "readings with a range of average speeds")

273 readings with a range of average speeds


#### **Converting Visibility to Metres**
Kilometres are multiplied by 1,000 and metres are kept as they are, so every value is in metres. `н/о` (not determined) stays empty. `км` is checked first, because `км` also contains the letter `м`.

In [59]:
visibility = []  # visibility in metres

for value in weather["visibility"]:  # go through every reading
    if value == "н/о":  # not determined
        visibility.append(None)
    elif "км" in value:  # kilometres, e.g. "10 км"
        visibility.append(float(value.replace("км", "").strip()) * 1000)
    else:  # metres, e.g. "500 м"
        visibility.append(float(value.replace("м", "").strip()))

weather["visibility"] = visibility  # replace the text with metres

#### **Converting Cloud to Total Cloud Amount**
Cloud is recorded in tenths of the sky covered (0–10). `ясно` (clear) is 0, `7 баллов` is 7, and in the detailed format (`10/10 300 м[Sc]`) the first number is the total cloud amount. `н/о` and `?/?` stay empty. The cell also prints any value that didn't fit one of these formats, so nothing is lost without me knowing.

In [60]:
cloud = []  # total cloud amount, 0-10
not_converted = []  # values that didn't match any rule

for value in weather["cloud"]:  # go through every reading
    if value == "ясно":  # clear sky
        cloud.append(0)
    elif "балл" in value:  # e.g. "7 баллов"
        cloud.append(float(value.split(" ")[0]))  # the number before the space
    elif "/" in value and value.split("/")[0].isdigit():  # e.g. "10/10 300 м[Sc]"
        cloud.append(float(value.split("/")[0]))  # the number before the /
    else:  # н/о, ?/? or anything unexpected
        cloud.append(None)
        if value != "н/о" and value != "?/?":  # not one of the expected unknowns
            not_converted.append(value)

weather["cloud"] = cloud  # replace the text with numbers
print("Unexpected cloud values:", not_converted)  # should be an empty list

Unexpected cloud values: []


#### **Turning Phenomena into Fog, Rain and Snow Flags**
The phenomena text is turned into three yes/no columns (1 = yes, 0 = no), using the glossary above. `in` checks whether a word appears anywhere in the text, including inside `{ }`. The original text is kept for checking.

In [62]:
fog = []  # 1 if fog was reported
rain = []  # 1 if rain, drizzle or freezing precipitation was reported
snow_flag = []  # 1 if snow or a snowstorm was reported

for value in weather["phenomena"]:  # go through every reading
    if "туман" in value:  # fog
        fog.append(1)
    else:
        fog.append(0)

    if "дождь" in value or "морось" in value or "осадки" in value:  # rain, drizzle, precipitation
        rain.append(1)
    else:
        rain.append(0)

    if "снег" in value or "метель" in value:  # snow, snowstorm
        snow_flag.append(1)
    else:
        snow_flag.append(0)

weather["fog"] = fog  # new columns
weather["rain"] = rain
weather["snow"] = snow_flag

#### **Checking the Converted Values**
I check that every column is now numeric and has sensible ranges, then answer three open questions: is the smallest gust always 10 m/s (so a blank gust means "below 10")? How often is visibility not recorded, and is humidity high at those times? And how many readings have visibility of 1 km or less but no fog in `phenomena`? I also check which times precipitation is reported at, which decides the daily rain rule in 2.11. `&` combines two conditions: both must be true.

In [63]:
print(weather.dtypes)  # every weather column should now be float or int
print(weather[["temp", "humidity", "wind_avg", "wind_gust", "visibility", "cloud", "precip", "snow_depth"]].describe())

print("Smallest gust:", weather["wind_gust"].min())  # expect 10
print("Visibility not determined:", weather["visibility"].isnull().sum(), "readings")
print(weather[weather["visibility"].isnull()]["humidity"].describe())  # humidity when visibility is missing

print("Readings with fog:", weather["fog"].sum())
low_vis_no_fog = weather[(weather["visibility"] <= 1000) & (weather["fog"] == 0)]  # 1 km or less, but no fog reported
print(len(low_vis_no_fog), "readings with visibility of 1 km or less but no fog")
print(low_vis_no_fog["phenomena"].value_counts().head(10))  # what phenomena says at those times

print(weather[weather["precip"].notnull()]["time_utc"].value_counts())  # which times precipitation is reported at

station        object
year            int64
month           int64
time_utc       object
date           object
visibility    float64
phenomena      object
cloud         float64
temp          float64
humidity      float64
temp_min      float64
temp_max      float64
precip        float64
snow_depth    float64
wind_avg      float64
wind_gust     float64
fog             int64
rain            int64
snow            int64
dtype: object
               temp      humidity      wind_avg    wind_gust    visibility  \
count  21461.000000  17736.000000  21457.000000  1744.000000  21380.000000   
mean      11.088556     67.676759      2.306637    11.801606  10131.973807   
std       10.673081     21.266735      1.930999     2.218339   4347.783356   
min      -21.100000     14.000000      0.000000    10.000000     50.000000   
25%        2.400000     51.000000      1.000000    10.000000  10000.000000   
50%       11.200000     73.000000      2.000000    12.000000  10000.000000   
75%       19.500000   

#### **Checking Precipitation**
Precipitation needs checking before it can be added up per day. One reading shows 320 mm, which is implausibly large. Precipitation is also reported at different times (mostly 03, 06, 15 and 18 UTC). The site's notes say hovering over a value shows the period it covers, so I check: which times each station reports at, and whether the period is stored in the page's HTML (which `get_text()` would have dropped).

In [64]:
print(weather[weather["precip"] > 50][["station", "year", "date", "time_utc", "precip", "phenomena"]])  # unusually large values

print(weather[weather["precip"].notnull()].groupby(["station", "time_utc"]).size())  # reporting times per station

for row in tables[1].find_all("tr")[1:]:  # weather rows of the saved March 2024 Donetsk page (from 2.3)
    cells = row.find_all("td")  # the 18 weather cells
    if cells[15].get_text().strip() != "":  # cell 15 is precipitation (R); find the first reading that has a value
        print(cells[15])  # the cell's full HTML, including any hidden attributes such as a title
        break  # stop after the first one

       station  year   date time_utc  precip phenomena
20558  kharkiv  2026  28.06       09   320.0          
station  time_utc
donetsk  03          437
         06          112
         09            1
         15          458
         18          112
kharkiv  00           67
         06          467
         09            1
         12           83
         15            1
         18          545
dtype: int64
<td class="temp_39" title="за 12 ч.">0.0</td>


#### **Recovering the Precipitation Periods**
Each precipitation value is the amount collected since the gauge was last emptied, not the rain at that moment. The page stores the period in a hidden `title` attribute (e.g. `title="за 12 ч."`, "over 12 hours"), which `get_text()` dropped. Donetsk reports mainly at 03 and 15 UTC and Kharkiv mainly at 06 and 18, but both also have values at other times. If those extra values cover overlapping periods, adding up a day's values would count the same rain twice. So I read the period of every value from the saved pages with `.get("title")`, and count which periods are used at each station and time, and in which years. No new requests are sent.

In [65]:
period_rows = []  # one entry per reading that has a precipitation value

for file_name in sorted(os.listdir("raw_html")):  # every saved page
    parts = file_name.replace(".html", "").split("_")  # e.g. ["kharkiv", "2026", "06"]
    station = parts[0]  # station name
    year = int(parts[1])  # year as a number

    f = open("raw_html/" + file_name, "rb")  # open the saved page
    page_soup = BeautifulSoup(f.read().decode("utf-8"), "html.parser")  # read, decode and make it searchable
    f.close()  # close the file

    page_tables = page_soup.find_all("table")  # the two tables
    time_rows = page_tables[0].find_all("tr")  # time/date rows
    weather_rows = page_tables[1].find_all("tr")  # weather rows

    for i in range(1, len(time_rows)):  # every reading, skipping the header
        time_cells = time_rows[i].find_all("td")  # time and date
        precip_cell = weather_rows[i].find_all("td")[15]  # cell 15 is precipitation (R)
        if precip_cell.get_text().strip() != "":  # only readings with a precipitation value
            period_rows.append([station, year,
                                time_cells[1].get_text().strip(),  # date
                                time_cells[0].get_text().strip(),  # time
                                precip_cell.get("title")])  # the hidden period, e.g. "за 12 ч."

periods = pd.DataFrame(period_rows, columns=["station", "year", "date", "time_utc", "precip_period"])  # one table
periods = periods.drop_duplicates(subset=key)  # the duplicate readings appear here too; keep one of each
print(periods.shape)  # should match the 2,284 precipitation values in the check above

print(periods.groupby(["station", "time_utc", "precip_period"]).size())  # which periods are used at each time
print(periods.groupby(["station", "year", "time_utc"]).size())  # which times are used in each year (did the schedule change?)

(2284, 5)
station  time_utc  precip_period
donetsk  03        за 12 ч.         437
         06        за 12 ч.         112
         09        за 12 ч.           1
         15        за 12 ч.         458
         18        за 12 ч.         112
kharkiv  00        за 6 ч.           67
         06        за 12 ч.         467
         09        за 1 ч.            1
         12        за 6 ч.           83
         15        за 6 ч.            1
         18        за 12 ч.         545
dtype: int64
station  year  time_utc
donetsk  2023  03          104
               06          112
               15          117
               18          112
         2024  03          106
               15          108
         2025  03          118
               15          126
         2026  03          109
               09            1
               15          107
kharkiv  2023  00           64
               06          133
               12           67
               18          147
         2024  

#### **Checking Donetsk's 2023 Schedule**
In 2023 Donetsk reported 12-hour totals at 03 and 15 UTC and also at 06 and 18 UTC. If both pairs were used on the same days, their periods overlap and the rain would be counted twice. If the station switched from one pair to the other during the year, there is no overlap. I count each reporting time by month, and count the days that have values from both pairs. `|` means "or" when combining conditions, and `&` between two sets keeps only the dates found in both.

In [66]:
months = []  # the month of each precipitation reading
for d in periods["date"]:  # dates look like "14.05"
    months.append(int(d.split(".")[1]))  # the part after the dot is the month
periods["month"] = months  # add as a column

donetsk_2023 = periods[(periods["station"] == "donetsk") & (periods["year"] == 2023)]  # Donetsk, 2023 only
print(donetsk_2023.groupby(["month", "time_utc"]).size())  # which times were used in each month

pair_03_15 = donetsk_2023[(donetsk_2023["time_utc"] == "03") | (donetsk_2023["time_utc"] == "15")]  # main pair
pair_06_18 = donetsk_2023[(donetsk_2023["time_utc"] == "06") | (donetsk_2023["time_utc"] == "18")]  # 2023-only pair
dates_03_15 = set(pair_03_15["date"])  # each date once
dates_06_18 = set(pair_06_18["date"])  # each date once
print(len(dates_03_15), "days with 03/15 values |", len(dates_06_18), "days with 06/18 values |",
      len(dates_03_15 & dates_06_18), "days with both")  # & keeps dates in both sets

month  time_utc
1      03          11
       06           9
       15          14
       18          12
2      03          15
       06          17
       15          16
       18          14
3      03          11
       06          10
       15           9
       18           8
4      03          18
       06          20
       15          12
       18          19
5      03           9
       06           8
       15           8
       18          10
8      06           1
       15           7
       18           5
9      03           3
       06           3
       15           2
       18           3
10     03           6
       06          10
       15          12
       18           7
11     03          12
       06          15
       15          16
       18          15
12     03          19
       06          19
       15          21
       18          19
dtype: int64
156 days with 03/15 values | 153 days with 06/18 values | 145 days with both


#### **Fixing Precipitation and Dropping Humidity**
**Precipitation:** Each station's values are kept only at its main pair of times, where both are 12-hour totals 12 hours apart, so together they cover exactly one day with no overlap: 03 and 15 UTC at Donetsk, 06 and 18 UTC at Kharkiv. Every other value is set to empty: Donetsk's extra 12-hour values at 06 and 18 in 2023 (which overlap the main pair on 145 days), Kharkiv's 6-hour values (which fall inside its 12-hour ones), and the 320 mm value, which the page records as falling in a single hour.

**Humidity:** visibility is missing in only 81 readings, and humidity at those times has a median of 75.5%, well below the near-100% that comes with fog. So humidity wouldn't help identify fog, and it is dropped.

The loop uses `weather.index`, the original row numbers, because 13 rows were removed in 2.10.1 and the numbers are no longer continuous. `pd.notnull()` is True when a value is not empty.

In [67]:
precip_clean = []  # precipitation, kept only at each station's main pair of times
removed = 0  # counts values that are set to empty

for i in weather.index:  # every reading, using its original row number
    station = weather["station"][i]  # which station
    time = weather["time_utc"][i]  # reading time, e.g. "03"
    value = weather["precip"][i]  # the precipitation value (or empty)

    if station == "donetsk" and (time == "03" or time == "15"):  # Donetsk's main 12-hour pair
        precip_clean.append(value)  # keep it
    elif station == "kharkiv" and (time == "06" or time == "18"):  # Kharkiv's main 12-hour pair
        precip_clean.append(value)  # keep it
    else:  # any other time
        precip_clean.append(None)  # set to empty
        if pd.notnull(value):  # it had a value that is now removed
            removed = removed + 1  # count it

weather["precip"] = precip_clean  # replace the column
print(removed, "precipitation values removed")  # expect 377
print(weather["precip"].notnull().sum(), "precipitation values kept")  # expect 1,907
print(weather["precip"].describe())  # the largest value should now be realistic

if "humidity" in weather.columns:  # only drop it if it's still there
    weather = weather.drop(columns=["humidity"])  # see the reason above
print(list(weather.columns))  # the final columns

377 precipitation values removed
1907 precipitation values kept
count    1907.000000
mean        1.976927
std         3.710480
min         0.000000
25%         0.000000
50%         0.500000
75%         2.000000
max        34.000000
Name: precip, dtype: float64
['station', 'year', 'month', 'time_utc', 'date', 'visibility', 'phenomena', 'cloud', 'temp', 'temp_min', 'temp_max', 'precip', 'snow_depth', 'wind_avg', 'wind_gust', 'fog', 'rain', 'snow']


#### **Observations on the Conversion**
**All values are now numbers.** Every weather column is numeric, apart from `phenomena`, which is kept as text for checking. No cloud value fell outside the conversion rules, so nothing was lost without being noticed.

**Wind:** 273 readings give the average speed as a range (e.g. `6-10`), recorded as the midpoint. Gusts appear in only 1,744 readings (8%), and the smallest gust in the full data is 10 m/s, which confirms on all 21,461 readings what March suggested: a blank gust means the gust was below 10 m/s, not that it is missing.

**Visibility and humidity:** Visibility is missing (`н/о`) in only 81 readings (0.4%). At those times humidity has a median of 75.5%, well below the near-100% that comes with fog, so humidity wouldn't help identify fog, and I dropped it.

**Fog:** 683 readings report fog. Another 202 have visibility of 1 km or less with no fog in `phenomena`. Most of these are explained by rain, drizzle or snow (which also lower visibility) or by haze, but 50 have a blank `phenomena` and may be unreported fog. So the fog flag and low visibility measure slightly different things, and in 2.11 I keep both: a daily fog flag and the day's minimum visibility.

**Snow depth:** only 325 readings have a value, and the minimum is 0, so a blank means "not measured" rather than "no snow". This is too sparse to use day by day, so the snow flag and temperature are used for the frozen-ground hypothesis instead.

**Precipitation:**
- One reading showed 320 mm. Its period (hidden in the page's HTML) was one hour, which is more than the world record for an hour's rainfall, and no rain was reported at the time, so it is an error.
- Each value is the total since the gauge was last emptied, and the hidden periods show the stations work differently. Donetsk reports 12-hour totals at 03 and 15 UTC; Kharkiv reports 12-hour totals at 06 and 18 UTC, plus 6-hour totals at 00 and 12, mostly in 2023.
- In 2023 Donetsk also reported 12-hour totals at 06 and 18, on 145 of the same days as 03 and 15, so their periods overlap.
- Adding every value would have counted the same rain twice, so I kept only each station's main pair of 12-hour totals, which together cover one day without overlap. 377 values were removed and 1,907 kept, and the largest is now 34 mm, a realistic 12-hour total.

### **2.11 - One Row per Day**
The Kaggle data has one row per day, so the 3-hourly readings are combined into one row per station per day. This section has three steps:

**1.** Work out each reading's local date

**2.** Combine the readings into days

**3.** Put both stations side by side and save the daily table

#### **2.11.1 - Working Out Each Reading's Local Date**
Times on the site are UTC, but the losses are counted by Ukrainian reporting days, so each reading is moved to local time before it is given a day. I use UTC+3 for both stations all year: that is Kyiv summer time, and one hour ahead in winter. This only moves the 21:00 UTC reading on winter days, and the loss data's own day boundary is approximate, so handling the clock change wouldn't add real precision.

I build each reading's full UTC date and time as text (e.g. `2024-03-02 21:00`), turn it into a real date with `pd.to_datetime()`, and add 3 hours with `pd.Timedelta(hours=3)`. `.dt.normalize()` then sets the time to 00:00, keeping only the day.

In [71]:
utc_times = []  # each reading's UTC date and time, as text

for i in weather.index:  # every reading, by its original row number
    day = weather["date"][i].split(".")[0]  # "2.03" -> "2"
    month = weather["date"][i].split(".")[1]  # "2.03" -> "03"
    year = str(weather["year"][i])  # 2024 -> "2024"
    utc_times.append(year + "-" + month + "-" + day.zfill(2) + " " + weather["time_utc"][i] + ":00")  # e.g. "2024-03-02 21:00"

weather["time_local"] = pd.to_datetime(utc_times, format="%Y-%m-%d %H:%M") + pd.Timedelta(hours=3)  # UTC + 3 hours
weather["date_local"] = weather["time_local"].dt.normalize()  # the local day, with the time set to 00:00

# The first 9 readings (1 Jan 00:00 UTC to 2 Jan 00:00 UTC): each local time should be 3 hours later,
# and the 21:00 UTC reading on 1 Jan should become 00:00 on 2 Jan
print(weather[["station", "date", "time_utc", "time_local", "date_local"]].head(9))

   station  date time_utc          time_local date_local
0  donetsk  1.01       00 2023-01-01 03:00:00 2023-01-01
1  donetsk  1.01       03 2023-01-01 06:00:00 2023-01-01
2  donetsk  1.01       06 2023-01-01 09:00:00 2023-01-01
3  donetsk  1.01       09 2023-01-01 12:00:00 2023-01-01
4  donetsk  1.01       12 2023-01-01 15:00:00 2023-01-01
5  donetsk  1.01       15 2023-01-01 18:00:00 2023-01-01
6  donetsk  1.01       18 2023-01-01 21:00:00 2023-01-01
7  donetsk  1.01       21 2023-01-02 00:00:00 2023-01-02
8  donetsk  2.01       00 2023-01-02 03:00:00 2023-01-02


#### **2.11.2 - Combining Readings into Days**
For each station and local day, I combine the readings using the rules below. `.agg()` applies a different rule to each column in one step.

| Daily column | Rule | Hypothesis it serves |
|---|---|---|
| `readings` | number of readings that day | data quality |
| `temp_mean` | mean temperature | freezing and thaw |
| `temp_min_day` | lowest of the readings and the station's minimum temperature | freezing |
| `wind_mean` | mean average wind speed | wind |
| `wind_max` | strongest wind that day: the gust where there was one, otherwise the average | wind and drones |
| `vis_min` | lowest visibility | fog and low visibility |
| `cloud_mean` | mean total cloud | low cloud |
| `precip_day` | sum of the day's 12-hour precipitation totals | rain and mud |
| `precip_reports` | how many of those totals the day has (0, 1 or 2) | data quality |
| `fog_day`, `rain_day`, `snow_day` | 1 if any reading that day has the flag | fog, rain, snow |

Under UTC+3, both of each station's 12-hour totals fall on the same local day (Donetsk 03 and 15 UTC = 06:00 and 18:00; Kharkiv 06 and 18 UTC = 09:00 and 21:00), so adding them gives that day's total. If a day has no precipitation value at all, `precip_day` is left empty rather than set to 0, because a missing value doesn't prove there was no rain. Finally, days with fewer than 6 of their 8 readings are dropped, because a mean or minimum from only a few readings could miss most of the day.

`.min(axis=1)` and `.max(axis=1)` compare columns across each row (rather than down a column), ignoring empty values.

In [72]:
weather["low_temp"] = weather[["temp", "temp_min"]].min(axis=1)  # the lower of the reading's temperature and the station minimum
weather["strongest_wind"] = weather[["wind_avg", "wind_gust"]].max(axis=1)  # the gust if there was one, otherwise the average

daily = weather.groupby(["station", "date_local"]).agg(
    readings=("time_utc", "count"),  # how many readings that day
    temp_mean=("temp", "mean"),  # mean temperature
    temp_min_day=("low_temp", "min"),  # lowest temperature
    wind_mean=("wind_avg", "mean"),  # mean average wind
    wind_max=("strongest_wind", "max"),  # strongest wind
    vis_min=("visibility", "min"),  # lowest visibility
    cloud_mean=("cloud", "mean"),  # mean cloud cover
    precip_reports=("precip", "count"),  # how many 12-hour totals (0, 1 or 2)
    precip_day=("precip", "sum"),  # total precipitation
    fog_day=("fog", "max"),  # 1 if any reading had fog
    rain_day=("rain", "max"),  # 1 if any reading had rain
    snow_day=("snow", "max")).reset_index()  # 1 if any reading had snow; reset_index makes station and date normal columns

precip_fixed = []  # precip_day, but empty where there was no value at all
for i in daily.index:  # every day
    if daily["precip_reports"][i] == 0:  # no precipitation value that day
        precip_fixed.append(None)  # leave empty, not 0
    else:
        precip_fixed.append(daily["precip_day"][i])  # keep the total
daily["precip_day"] = precip_fixed  # replace the column

print(daily.shape)  # days before dropping
print(daily[daily["readings"] < 6].groupby("station").size())  # days with fewer than 6 readings, per station
daily = daily[daily["readings"] >= 6]  # keep only days with at least 6 readings
print(daily.shape)  # days kept

(2718, 14)
station
donetsk    23
kharkiv     1
dtype: int64
(2694, 14)


#### **Checking the Daily Table**
I check how many days each station has, the range of each daily value, and how many days report rain in `phenomena` but have no precipitation value. That last count shows whether a blank precipitation value can be treated as "no rain".

In [73]:
print(daily.groupby("station").size())  # days kept per station
print(daily.groupby("station")["date_local"].min(), daily.groupby("station")["date_local"].max())  # first and last day per station
print(daily.describe())  # ranges of every daily value

rain_no_value = daily[(daily["rain_day"] == 1) & (daily["precip_reports"] == 0)]  # rain reported, but no amount recorded
print(len(rain_no_value), "days with rain in phenomena but no precipitation value")

station
donetsk    1305
kharkiv    1389
dtype: int64
station
donetsk   2023-01-01
kharkiv   2023-01-01
Name: date_local, dtype: datetime64[ns] station
donetsk   2026-10-20
kharkiv   2026-10-20
Name: date_local, dtype: datetime64[ns]
                          date_local     readings    temp_mean  temp_min_day  \
count                           2694  2694.000000  2694.000000   2694.000000   
mean   2024-12-10 07:14:33.942093568     7.932814    11.096990      6.639310   
min              2023-01-01 00:00:00     6.000000   -17.250000    -21.900000   
25%              2024-01-12 00:00:00     8.000000     2.575000      0.100000   
50%              2024-12-15 00:00:00     8.000000    11.687500      7.200000   
75%              2025-11-17 00:00:00     8.000000    19.812500     14.000000   
max              2026-10-20 00:00:00     8.000000    32.125000     25.700000   
std                              NaN     0.292777    10.131268      8.920006   

         wind_mean     wind_max       vis_min 

#### **2.11.3 - Putting the Stations Side by Side and Saving**
The merge in Step 3 needs one row per date, so I split the table by station, add the station name to the front of each column (`.add_prefix()`, e.g. `donetsk_temp_mean`), and join the two stations on the date. `how="outer"` keeps days that only one station has, such as the Donetsk gap in June–July 2023. The result is saved as `weather_daily.csv` (`.to_csv()`, with `index=False` so the row numbers aren't saved). This file, not the raw pages, is what goes on GitHub, with the source credited in the README (clause 3.7).

In [74]:
donetsk_daily = daily[daily["station"] == "donetsk"].drop(columns=["station"])  # Donetsk days only
donetsk_daily = donetsk_daily.add_prefix("donetsk_")  # e.g. temp_mean -> donetsk_temp_mean
donetsk_daily = donetsk_daily.rename(columns={"donetsk_date_local": "date"})  # the date column keeps a plain name

kharkiv_daily = daily[daily["station"] == "kharkiv"].drop(columns=["station"])  # Kharkiv days only
kharkiv_daily = kharkiv_daily.add_prefix("kharkiv_")  # e.g. temp_mean -> kharkiv_temp_mean
kharkiv_daily = kharkiv_daily.rename(columns={"kharkiv_date_local": "date"})  # same plain date name

weather_daily = pd.merge(donetsk_daily, kharkiv_daily, on="date", how="outer")  # one row per date, both stations
weather_daily = weather_daily.sort_values("date").reset_index(drop=True)  # oldest first

print(weather_daily.shape)  # (days, columns)
print(weather_daily["date"].min(), "to", weather_daily["date"].max())  # date range
weather_daily.to_csv("weather_daily.csv", index=False)  # save to the Drive Data folder

(1389, 25)
2023-01-01 00:00:00 to 2026-10-20 00:00:00


#### **Filling Dry Days**
Only 1,308 of the 2,694 days have a precipitation amount. When rain was reported in `phenomena`, an amount was recorded on all but 27 of about 946 days, so a missing amount almost always means nothing fell. I therefore set the amount to 0 mm on days with no amount **and** no rain or snow reported. Days where rain or snow was reported but no amount was recorded stay empty, because their amount is unknown. `pd.isnull()` is True when a value is empty.

In [75]:
precip_filled = []  # precip_day with dry days set to 0
filled = 0  # counts days set to 0

for i in daily.index:  # every day
    value = daily["precip_day"][i]  # that day's total (or empty)
    if pd.isnull(value) and daily["rain_day"][i] == 0 and daily["snow_day"][i] == 0:  # no amount, no rain, no snow
        precip_filled.append(0.0)  # a dry day
        filled = filled + 1  # count it
    else:
        precip_filled.append(value)  # keep the amount, or leave empty if rain/snow fell but no amount was recorded

daily["precip_day"] = precip_filled  # replace the column
print(filled, "dry days set to 0 mm")
print(daily["precip_day"].isnull().sum(), "days still without an amount (rain or snow reported, amount unknown)")

1356 dry days set to 0 mm
30 days still without an amount (rain or snow reported, amount unknown)


#### **Observations on the Daily Table**
**Local dates:** Moving every reading to UTC+3 works as intended: the 21:00 UTC reading becomes 00:00 the next day, so a reading the site lists under 1 January belongs to 2 January.

**Days kept:** The readings formed 2,718 station-days. 24 had fewer than 6 readings and were dropped (23 at Donetsk, 1 at Kharkiv); some of these are single readings that the time shift pushed into a new day, such as 21 October 2026. Kharkiv keeps all 1,389 days from 1 January 2023 to 20 October 2026; Donetsk keeps 1,305, because of the June–July 2023 gap and its short days.

**Values:** Daily mean temperature ranges from −17.3 to 32.1 °C, the strongest wind reaches 24 m/s and the lowest visibility is 50 m, all realistic. Across both stations, rain falls on 35% of days, snow on 12% and fog on 7%.

**Precipitation:** Only 1,308 days had a precipitation amount, but rain was reported without an amount on only 27 of about 946 rainy days, so a missing amount almost always means a dry day. I set [FILLED] dry days to 0 mm; [STILL EMPTY] days with rain or snow reported but no amount stay empty. Days with only one of the two 12-hour totals (`precip_reports` = 1) may undercount the day's rain, so `precip_reports` is kept to allow those days to be left out in Step 4.

**Result:** `weather_daily.csv` has one row per date (1,389 dates) and 25 columns: the date plus 12 daily values for each station.